In [1]:
import os
import re
import json
import unicodedata
from datetime import datetime

import pandas as pd

# =========================
# CẤU HÌNH ĐƯỜNG DẪN
# =========================
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath("D:\\TTTN\\Project\\scripts\\preprocessing.ipynb")))
RAW_INPUT_PATH = os.path.join(BASE_DIR, "data_topcv", "topcv_all_fields_merged_2026-03-16_20-57-23.xlsx")
OUTPUT_DIR = os.path.join(BASE_DIR, "data_processed")

print("BASE_DIR         :", BASE_DIR)
print("RAW_INPUT_PATH   :", RAW_INPUT_PATH)
print("OUTPUT_DIR       :", OUTPUT_DIR)
print("File tồn tại?"   , os.path.exists(RAW_INPUT_PATH))

BASE_DIR         : D:\TTTN\Project
RAW_INPUT_PATH   : D:\TTTN\Project\data_topcv\topcv_all_fields_merged_2026-03-16_20-57-23.xlsx
OUTPUT_DIR       : D:\TTTN\Project\data_processed
File tồn tại? True


In [2]:
# pd.set_option('display.max_columns', None)   # hiển thị tất cả cột
# pd.set_option('display.max_rows', None)      # hiển thị tất cả dòng
# pd.set_option('display.width', None)         # không giới hạn chiều rộng
# pd.set_option('display.max_colwidth', None)  # không cắt nội dung cột

# 1. Clean cơ bản

In [3]:
# Chuẩn hóa giá trị trống
def normalize_empty_value(val):
    if pd.isna(val):
        return None
    val_str = str(val).strip()
    if not val_str or val_str.lower() in {"nan", "none", "null", "n/a", "na"}:
        return None
    return val_str

# Lấy giá trị đầu tiên không rỗng từ danh sách
def first_non_empty(*values):
    for v in values:
        v = normalize_empty_value(v)
        if v is not None:
            return v
    return None

# Chuẩn hóa unicode
def normalize_unicode(text: str) -> str:
    if text is None:
        return ""
    return unicodedata.normalize("NFC", str(text))

# Làm sạch nhẹ: dùng cho PhoBERT / embedding
def clean_text_light(text: str) -> str:
    text = normalize_empty_value(text)
    if text is None:
        return ""

    text = normalize_unicode(text)

    # bỏ HTML
    text = re.sub(r"<[^>]+>", " ", text)

    # chuẩn hóa một số bullet/ký tự đặc biệt về dạng dễ đọc
    text = (
        text.replace("•", " - ")
            .replace("▪", " - ")
            .replace("✅", " - ")
            .replace("✔", " - ")
            .replace("★", " - ")
            .replace("−", "-")
            .replace("–", "-")
            .replace("—", "-")
    )

    # chuẩn hóa xuống dòng/tab
    text = re.sub(r"[\r\t]+", " ", text)
    text = re.sub(r"\n+", "\n", text)

    # bỏ zero-width / invisible chars
    text = re.sub(r"[\u200b-\u200f\uFEFF]", "", text)

    # chỉ loại các ký tự thật sự rác, giữ lại punctuation có ích
    text = re.sub(r"[^\w\sÀ-ỹ\.,;:/\-\+\#\(\)%]", " ", text)

    # chuẩn hóa khoảng trắng nhưng vẫn giữ cấu trúc cơ bản
    text = re.sub(r"[ ]{2,}", " ", text)
    text = re.sub(r" ?\n ?", "\n", text)

    return text.strip()

# Làm sạch chặt hơn: dùng cho regex / matching / extraction
def clean_text_strict(text: str) -> str:
    text = normalize_empty_value(text)
    if text is None:
        return ""

    text = normalize_unicode(text).lower()

    # bỏ HTML
    text = re.sub(r"<[^>]+>", " ", text)

    # chuẩn hóa bullet/ký hiệu
    text = (
        text.replace("•", " ")
            .replace("▪", " ")
            .replace("✅", " ")
            .replace("✔", " ")
            .replace("★", " ")
            .replace("−", "-")
            .replace("–", "-")
            .replace("—", "-")
    )

    # bỏ xuống dòng/tab
    text = re.sub(r"[\r\n\t]+", " ", text)

    # bỏ zero-width / invisible chars
    text = re.sub(r"[\u200b-\u200f\uFEFF]", "", text)

    # vẫn giữ một số ký hiệu quan trọng cho skill
    text = re.sub(r"[^\w\sÀ-ỹ\./\-\+\#]", " ", text)

    # chuẩn hóa khoảng trắng
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [4]:
# Load dữ liệu thô
def load_raw_data(input_path: str) -> pd.DataFrame:
    ext = os.path.splitext(input_path)[1].lower()
    if ext == ".xlsx":
        df = pd.read_excel(input_path)
    elif ext == ".csv":
        df = pd.read_csv(input_path)
    else:
        raise ValueError(f"Định dạng file không hỗ trợ: {ext}")
    
    print(f"[INFO] Loaded raw data: {df.shape[0]} rows x {df.shape[1]} cols")
    return df


# Thực thi
raw_df = load_raw_data(RAW_INPUT_PATH)

# Kiểm tra nhanh
display(raw_df.head(3))
print("\nColumns:\n", raw_df.columns.tolist())

[INFO] Loaded raw data: 325 rows x 33 cols


,job_url,source_field_name,field_count,title,detail_title,company_name,company_name_full,company_url,company_url_from_job,salary_list,...,desc_mota,desc_yeucau,desc_quyenloi,company_website,company_scale_from_job,company_scale,company_field_from_job,company_address_from_job,company_address,company_description
0,https://www.topcv.vn/brand/beeogisticscorporat...,Data Analyst,1,Junior FP&A Analyst - Hồ Chí Minh,Junior FP&A Analyst - Hồ Chí Minh,Bee Logistics Corporation,Bee Logistics Corporation,https://www.topcv.vn/brand/beeogisticscorporat...,https://www.topcv.vn/brand/beeogisticscorporat...,12 - 20 triệu,...,"– Overseeing all financial planning, reporting...",– At least 2 year’ experience in fthe inance/a...,– Competitive salary according to personal exp...,http://www.beelogistics.com/,NaN,500-1000,NaN,NaN,"Addr: 11th Floor, TTC Tower, 253 Hoang Van Thu...",Xuất phát với ước mơ xây dựng một doanh nghiệp...
1,https://www.topcv.vn/brand/educa/tuyen-dung/da...,Data Analyst,1,Data Governance Specialist,Data Governance Specialist,EDUPIA,EDUPIA,https://www.topcv.vn/brand/educa?id=17270,https://www.topcv.vn/brand/educa?id=17270,20 - 30 triệu,...,"− Xây dựng, triển khai và duy trì khung Data G...","− Đã tốt nghiệp ĐH trở lên, ưu tiên các chuyên...",Mức lương thỏa thuận theo năng lực. Thử việc h...,https://edupia.vn/,NaN,500-1000,NaN,NaN,"Trụ sở: Tầng 2,3,5,6 - Tòa Vincem Comatce Towe...","Được thành lập năm 2018, Edupia là Edtech lớn ..."
2,https://www.topcv.vn/brand/educa/tuyen-dung/pr...,AI Engineer,1,Project Manager Dự Án AI HUB,Project Manager Dự Án AI HUB,EDUPIA,EDUPIA,https://www.topcv.vn/brand/educa?id=17270,https://www.topcv.vn/brand/educa?id=17270,30 - 35 triệu,...,1. Lập kế hoạch & Xây dựng chiến lược AI (20%)...,Tối thiểu 2–3 năm kinh nghiệm ở vị trí Product...,Tinh thần: - Làm việc trong môi trường start-u...,https://edupia.vn/,NaN,500-1000,NaN,NaN,"Trụ sở: Tầng 2,3,5,6 - Tòa Vincem Comatce Towe...","Được thành lập năm 2018, Edupia là Edtech lớn ..."



Columns:
 ['job_url', 'source_field_name', 'field_count', 'title', 'detail_title', 'company_name', 'company_name_full', 'company_url', 'company_url_from_job', 'salary_list', 'detail_salary', 'address_list', 'detail_location', 'exp_list', 'detail_experience', 'deadline', 'tags', 'job_level', 'education_level', 'job_quantity', 'employment_type', 'working_addresses', 'working_times', 'desc_mota', 'desc_yeucau', 'desc_quyenloi', 'company_website', 'company_scale_from_job', 'company_scale', 'company_field_from_job', 'company_address_from_job', 'company_address', 'company_description']


In [5]:
# Gộp cột đại diện
def merge_semantic_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame()
    out["job_url"] = df.get("job_url")
    out["source_field_name"] = df.get("source_field_name")
    out["field_count"] = df.get("field_count")

    out["job_title_raw"] = [first_non_empty(a, b) for a, b in zip(df.get("detail_title"), df.get("title"))]
    
    out["salary_raw"] = [first_non_empty(a, b) for a, b in zip(df.get("detail_salary"), df.get("salary_list"))]
    out["location_raw"] = [first_non_empty(a, b) for a,b in zip(df.get("detail_location"), df.get("address_list"))]
    out["working_addresses_raw"] = df.get("working_addresses")
    out["working_times_raw"] = df.get("working_times")
    out["experience_raw"] = [first_non_empty(a, b) for a, b in zip(df.get("detail_experience"), df.get("exp_list"))]
    out["tags_raw"] = df.get("tags")
    
    out["job_level_raw"] = df.get("job_level")
    out["education_level_raw"] = df.get("education_level")
    out["employment_type_raw"] = df.get("employment_type")
    out["job_quantity"] = df.get("job_quantity")
    out["deadline_raw"] = df.get("deadline")
    
    out["description_raw"]  = df.get("desc_mota")
    out["requirements_raw"] = df.get("desc_yeucau")
    out["benefits_raw"]     = df.get("desc_quyenloi")

    out["company_name_raw"] = [first_non_empty(a, b) for a, b in zip(df.get("company_name_full"), df.get("company_name"))]
    out["company_url"] = [first_non_empty(a, b) for a, b in zip(df.get("company_url_from_job"), df.get("company_url"))]
    out["company_website_raw"] = df.get("company_website")
    out["company_scale_raw"] = [first_non_empty(a, b) for a, b in zip(df.get("company_scale_from_job"), df.get("company_scale"))]
    out["company_field_raw"] = df.get("company_field_from_job")
    out["company_address_raw"] = [first_non_empty(a, b) for a, b in zip(df.get("company_address_from_job"), df.get("company_address"))]
    out["company_description_raw"] = df.get("company_description")

    print(f"[INFO] After merging: {out.shape[0]} rows x {out.shape[1]} cols")
    return out


# Chạy
df = merge_semantic_columns(raw_df)
display(df.head(3))
df.info()

[INFO] After merging: 325 rows x 25 cols


,job_url,source_field_name,field_count,job_title_raw,salary_raw,location_raw,working_addresses_raw,working_times_raw,experience_raw,tags_raw,...,description_raw,requirements_raw,benefits_raw,company_name_raw,company_url,company_website_raw,company_scale_raw,company_field_raw,company_address_raw,company_description_raw
0,https://www.topcv.vn/brand/beeogisticscorporat...,Data Analyst,1,Junior FP&A Analyst - Hồ Chí Minh,12 - 20 triệu,Hồ Chí Minh,(đã được cập nhật theo Danh mục Hành chính mới...,Thứ 2 - Thứ 6 (từ 08:00 đến 17:30) Thứ 7 (từ 0...,2 năm,Chuyên môn Data Analyst; Tài chính; Kế toán,...,"– Overseeing all financial planning, reporting...",– At least 2 year’ experience in fthe inance/a...,– Competitive salary according to personal exp...,Bee Logistics Corporation,https://www.topcv.vn/brand/beeogisticscorporat...,http://www.beelogistics.com/,500-1000,NaN,"Addr: 11th Floor, TTC Tower, 253 Hoang Van Thu...",Xuất phát với ước mơ xây dựng một doanh nghiệp...
1,https://www.topcv.vn/brand/educa/tuyen-dung/da...,Data Analyst,1,Data Governance Specialist,20 - 30 triệu,Hà Nội,(đã được cập nhật theo Danh mục Hành chính mới...,Thứ 2 - Thứ 6 (từ 08:00 đến 17:30) Thứ 7 (từ 0...,2 năm,Chuyên môn Data Analyst,...,"− Xây dựng, triển khai và duy trì khung Data G...","− Đã tốt nghiệp ĐH trở lên, ưu tiên các chuyên...",Mức lương thỏa thuận theo năng lực. Thử việc h...,EDUPIA,https://www.topcv.vn/brand/educa?id=17270,https://edupia.vn/,500-1000,NaN,"Trụ sở: Tầng 2,3,5,6 - Tòa Vincem Comatce Towe...","Được thành lập năm 2018, Edupia là Edtech lớn ..."
2,https://www.topcv.vn/brand/educa/tuyen-dung/pr...,AI Engineer,1,Project Manager Dự Án AI HUB,30 - 35 triệu,Hà Nội,(đã được cập nhật theo Danh mục Hành chính mới...,Thứ 2 - Thứ 6 (từ 08:00 đến 17:30) Thứ 7 (từ 0...,2 năm,Chuyên môn IT Project Manager,...,1. Lập kế hoạch & Xây dựng chiến lược AI (20%)...,Tối thiểu 2–3 năm kinh nghiệm ở vị trí Product...,Tinh thần: - Làm việc trong môi trường start-u...,EDUPIA,https://www.topcv.vn/brand/educa?id=17270,https://edupia.vn/,500-1000,NaN,"Trụ sở: Tầng 2,3,5,6 - Tòa Vincem Comatce Towe...","Được thành lập năm 2018, Edupia là Edtech lớn ..."


<class 'pandas.DataFrame'>
RangeIndex: 325 entries, 0 to 324
Data columns (total 25 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   job_url                  325 non-null    str  
 1   source_field_name        325 non-null    str  
 2   field_count              325 non-null    int64
 3   job_title_raw            325 non-null    str  
 4   salary_raw               325 non-null    str  
 5   location_raw             325 non-null    str  
 6   working_addresses_raw    325 non-null    str  
 7   working_times_raw        294 non-null    str  
 8   experience_raw           325 non-null    str  
 9   tags_raw                 325 non-null    str  
 10  job_level_raw            325 non-null    str  
 11  education_level_raw      325 non-null    str  
 12  employment_type_raw      325 non-null    str  
 13  job_quantity             325 non-null    str  
 14  deadline_raw             325 non-null    str  
 15  description_raw  

- Bắt đầu clean raw data từ đây

In [6]:
# Tạo bản clean riêng để không làm mất dữ liệu raw
df_clean = df.copy()

print(f"[INFO] df_raw shape   : {df.shape}")
print(f"[INFO] df_clean shape : {df_clean.shape}")

[INFO] df_raw shape   : (325, 25)
[INFO] df_clean shape : (325, 25)


In [7]:
# Kiểm tra nhanh tỷ lệ thiếu dữ liệu ở các cột quan trọng
key_cols = [
    "job_title_raw",
    "salary_raw",
    "location_raw",
    "working_addresses_raw",
    "working_times_raw",
    "experience_raw",
    "tags_raw",
    "job_level_raw",
    "education_level_raw",
    "employment_type_raw",
    "description_raw",
    "requirements_raw",
    "benefits_raw",
    "company_name_raw",
    "company_description_raw"
]

missing_report = pd.DataFrame({
    "missing_count": df_clean[key_cols].isna().sum(),
    "missing_ratio": (df_clean[key_cols].isna().sum() / len(df_clean)).round(4)
}).sort_values("missing_ratio", ascending=False)

display(missing_report)

,missing_count,missing_ratio
working_times_raw,31,0.0954
company_description_raw,9,0.0277
salary_raw,0,0.0000
location_raw,0,0.0000
working_addresses_raw,0,0.0000
experience_raw,0,0.0000
job_title_raw,0,0.0000
tags_raw,0,0.0000
job_level_raw,0,0.0000
employment_type_raw,0,0.0000


In [8]:
# Chuẩn hóa giá trị rỗng cho toàn bộ các cột text/raw quan trọng
raw_text_cols = [
    "job_title_raw",
    "salary_raw",
    "location_raw",
    "working_addresses_raw",
    "working_times_raw",
    "experience_raw",
    "tags_raw",
    "job_level_raw",
    "education_level_raw",
    "employment_type_raw",
    "deadline_raw",
    "description_raw",
    "requirements_raw",
    "benefits_raw",
    "company_name_raw",
    "company_url",
    "company_website_raw",
    "company_scale_raw",
    "company_field_raw",
    "company_address_raw",
    "company_description_raw"
]

for col in raw_text_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].apply(normalize_empty_value)

print("[INFO] Đã chuẩn hóa giá trị rỗng cho các cột raw.")

[INFO] Đã chuẩn hóa giá trị rỗng cho các cột raw.


In [9]:
# Nhóm cột dùng clean nhẹ (giữ ngữ nghĩa cho PhoBERT / chatbot)
light_clean_cols = [
    "job_title_raw",
    "description_raw",
    "requirements_raw",
    "benefits_raw",
    "company_description_raw",
    "working_times_raw",
    "working_addresses_raw",
    "company_address_raw",
    "company_name_raw",
    "tags_raw"
]
# Tạo các cột clean nhẹ
for col in light_clean_cols:
    if col in df_clean.columns:
        new_col = col.replace("_raw", "_clean_light")
        df_clean[new_col] = df_clean[col].apply(clean_text_light)

print("[INFO] Đã tạo xong các cột *_clean_light")


[INFO] Đã tạo xong các cột *_clean_light


In [10]:
# Nhóm cột dùng clean chặt hơn (phục vụ normalize / regex / extract)
strict_clean_cols = [
    "job_title_raw",
    "salary_raw",
    "location_raw",
    "experience_raw",
    "education_level_raw",
    "employment_type_raw",
    "job_level_raw",
    "tags_raw",
    "company_field_raw",
    "working_times_raw",
    "working_addresses_raw"
]

# Tạo các cột clean chặt
for col in strict_clean_cols:
    if col in df_clean.columns:
        new_col = col.replace("_raw", "_clean_strict")
        df_clean[new_col] = df_clean[col].apply(clean_text_strict)

print("[INFO] Đã tạo xong các cột *_clean_strict")

[INFO] Đã tạo xong các cột *_clean_strict


In [11]:
clean_cols_created = [c for c in df_clean.columns if c.endswith("_clean_light") or c.endswith("_clean_strict")]
print(f"[INFO] Số cột clean đã tạo: {len(clean_cols_created)}")
clean_cols_created

[INFO] Số cột clean đã tạo: 21


['job_title_clean_light',
 'description_clean_light',
 'requirements_clean_light',
 'benefits_clean_light',
 'company_description_clean_light',
 'working_times_clean_light',
 'working_addresses_clean_light',
 'company_address_clean_light',
 'company_name_clean_light',
 'tags_clean_light',
 'job_title_clean_strict',
 'salary_clean_strict',
 'location_clean_strict',
 'experience_clean_strict',
 'education_level_clean_strict',
 'employment_type_clean_strict',
 'job_level_clean_strict',
 'tags_clean_strict',
 'company_field_clean_strict',
 'working_times_clean_strict',
 'working_addresses_clean_strict']

In [12]:
preview_cols = [
    "job_title_raw", "job_title_clean_light", "job_title_clean_strict",
    "description_raw", "description_clean_light",
    "requirements_raw", "requirements_clean_light",
    "location_raw", "location_clean_strict",
    "experience_raw", "experience_clean_strict",
    "tags_raw", "tags_clean_light", "tags_clean_strict"
]

preview_cols = [c for c in preview_cols if c in df_clean.columns]

display(df_clean[preview_cols].head(5))

,job_title_raw,job_title_clean_light,job_title_clean_strict,description_raw,description_clean_light,requirements_raw,requirements_clean_light,location_raw,location_clean_strict,experience_raw,experience_clean_strict,tags_raw,tags_clean_light,tags_clean_strict
0,Junior FP&A Analyst - Hồ Chí Minh,Junior FP A Analyst - Hồ Chí Minh,junior fp a analyst - hồ chí minh,"– Overseeing all financial planning, reporting...","- Overseeing all financial planning, reporting...",– At least 2 year’ experience in fthe inance/a...,- At least 2 year experience in fthe inance/ac...,Hồ Chí Minh,hồ chí minh,2 năm,2 năm,Chuyên môn Data Analyst; Tài chính; Kế toán,Chuyên môn Data Analyst; Tài chính; Kế toán,chuyên môn data analyst tài chính kế toán
1,Data Governance Specialist,Data Governance Specialist,data governance specialist,"− Xây dựng, triển khai và duy trì khung Data G...","- Xây dựng, triển khai và duy trì khung Data G...","− Đã tốt nghiệp ĐH trở lên, ưu tiên các chuyên...","- Đã tốt nghiệp ĐH trở lên, ưu tiên các chuyên...",Hà Nội,hà nội,2 năm,2 năm,Chuyên môn Data Analyst,Chuyên môn Data Analyst,chuyên môn data analyst
2,Project Manager Dự Án AI HUB,Project Manager Dự Án AI HUB,project manager dự án ai hub,1. Lập kế hoạch & Xây dựng chiến lược AI (20%)...,1. Lập kế hoạch Xây dựng chiến lược AI (20%) X...,Tối thiểu 2–3 năm kinh nghiệm ở vị trí Product...,Tối thiểu 2-3 năm kinh nghiệm ở vị trí Product...,Hà Nội,hà nội,2 năm,2 năm,Chuyên môn IT Project Manager,Chuyên môn IT Project Manager,chuyên môn it project manager
3,AI Engineer,AI Engineer,ai engineer,Xây dựng các capability AI và data processing ...,Xây dựng các capability AI và data processing ...,Kinh nghiệm AI/ML ứng dụng thực tế Python và x...,Kinh nghiệm AI/ML ứng dụng thực tế Python và x...,Hà Nội,hà nội,3 năm,3 năm,Chuyên môn AI Engineer; IT - Phần mềm,Chuyên môn AI Engineer; IT - Phần mềm,chuyên môn ai engineer it - phần mềm
4,AI Engineer,AI Engineer,ai engineer,Nghiên cứu và triển khai các giải pháp công ng...,Nghiên cứu và triển khai các giải pháp công ng...,"Tốt nghiệp Đại học chuyên ngành CNTT, Khoa học...","Tốt nghiệp Đại học chuyên ngành CNTT, Khoa học...",Hà Nội,hà nội,2 năm,2 năm,Chuyên môn AI Engineer,Chuyên môn AI Engineer,chuyên môn ai engineer


In [13]:
# Thống kê số bản ghi rỗng sau clean ở các cột chính
post_clean_check_cols = [
    "job_title_clean_light",
    "description_clean_light",
    "requirements_clean_light",
    "benefits_clean_light",
    "location_clean_strict",
    "experience_clean_strict",
    "tags_clean_light"
]

post_clean_check_cols = [c for c in post_clean_check_cols if c in df_clean.columns]

post_clean_report = pd.DataFrame({
    "empty_count": [(df_clean[c].fillna("").str.strip() == "").sum() for c in post_clean_check_cols],
    "empty_ratio": [round((df_clean[c].fillna("").str.strip() == "").mean(), 4) for c in post_clean_check_cols]
}, index=post_clean_check_cols)

display(post_clean_report)

,empty_count,empty_ratio
job_title_clean_light,0,0.0
description_clean_light,0,0.0
requirements_clean_light,0,0.0
benefits_clean_light,0,0.0
location_clean_strict,0,0.0
experience_clean_strict,0,0.0
tags_clean_light,0,0.0


In [14]:
# Kiểm tra độ dài text sau clean cho các cột quan trọng
length_check_cols = [
    "job_title_clean_light",
    "description_clean_light",
    "requirements_clean_light",
    "benefits_clean_light"
]

for col in length_check_cols:
    if col in df_clean.columns:
        df_clean[f"{col}_len"] = df_clean[col].fillna("").str.len()

length_summary = {}
for col in length_check_cols:
    len_col = f"{col}_len"
    if len_col in df_clean.columns:
        length_summary[col] = {
            "min_len": int(df_clean[len_col].min()),
            "max_len": int(df_clean[len_col].max()),
            "mean_len": round(df_clean[len_col].mean(), 2),
            "median_len": round(df_clean[len_col].median(), 2)
        }

length_summary = pd.DataFrame(length_summary).T
display(length_summary)

,min_len,max_len,mean_len,median_len
job_title_clean_light,8.0,161.0,35.20,31.0
description_clean_light,149.0,6034.0,1018.49,821.0
requirements_clean_light,118.0,5829.0,858.84,708.0
benefits_clean_light,90.0,3461.0,728.77,614.0


In [15]:
# Checkpoint sau bước cleaning
print(f"[INFO] df_clean shape after cleaning: {df_clean.shape}")
df_clean.head(3)

[INFO] df_clean shape after cleaning: (325, 50)


,job_url,source_field_name,field_count,job_title_raw,salary_raw,location_raw,working_addresses_raw,working_times_raw,experience_raw,tags_raw,...,employment_type_clean_strict,job_level_clean_strict,tags_clean_strict,company_field_clean_strict,working_times_clean_strict,working_addresses_clean_strict,job_title_clean_light_len,description_clean_light_len,requirements_clean_light_len,benefits_clean_light_len
0,https://www.topcv.vn/brand/beeogisticscorporat...,Data Analyst,1,Junior FP&A Analyst - Hồ Chí Minh,12 - 20 triệu,Hồ Chí Minh,(đã được cập nhật theo Danh mục Hành chính mới...,Thứ 2 - Thứ 6 (từ 08:00 đến 17:30) Thứ 7 (từ 0...,2 năm,Chuyên môn Data Analyst; Tài chính; Kế toán,...,toàn thời gian,nhân viên,chuyên môn data analyst tài chính kế toán,,thứ 2 - thứ 6 từ 08 00 đến 17 30 thứ 7 từ 08 0...,đã được cập nhật theo danh mục hành chính mới ...,33,1107,602,405
1,https://www.topcv.vn/brand/educa/tuyen-dung/da...,Data Analyst,1,Data Governance Specialist,20 - 30 triệu,Hà Nội,(đã được cập nhật theo Danh mục Hành chính mới...,Thứ 2 - Thứ 6 (từ 08:00 đến 17:30) Thứ 7 (từ 0...,2 năm,Chuyên môn Data Analyst,...,toàn thời gian,nhân viên,chuyên môn data analyst,,thứ 2 - thứ 6 từ 08 00 đến 17 30 thứ 7 từ 08 0...,đã được cập nhật theo danh mục hành chính mới ...,26,727,480,1509
2,https://www.topcv.vn/brand/educa/tuyen-dung/pr...,AI Engineer,1,Project Manager Dự Án AI HUB,30 - 35 triệu,Hà Nội,(đã được cập nhật theo Danh mục Hành chính mới...,Thứ 2 - Thứ 6 (từ 08:00 đến 17:30) Thứ 7 (từ 0...,2 năm,Chuyên môn IT Project Manager,...,toàn thời gian,nhân viên,chuyên môn it project manager,,thứ 2 - thứ 6 từ 08 00 đến 17 30 thứ 7 từ 08 0...,đã được cập nhật theo danh mục hành chính mới ...,28,3635,1753,2007


# - NORMALIZE METADATA

In [16]:
# =========================
# NORMALIZE JOB TITLE
# =========================

JOB_TITLE_REMOVE_PATTERNS = [
    r"\(.*?\)",                     # bỏ phần trong ngoặc
    r"\[.*?\]",                     # bỏ phần trong ngoặc vuông
    r"\s*-\s*(hà nội|ha noi|hồ chí minh|ho chi minh|đà nẵng|da nang|remote|hybrid).*$",
    r"\s*\|\s*.*$",                 # bỏ phần sau dấu |
]

JOB_TITLE_SYNONYM_MAP = {
    "data analyst": "data analyst",
    "chuyên viên phân tích dữ liệu": "data analyst",
    "nhân viên phân tích dữ liệu": "data analyst",
    "business analyst": "business analyst",
    "data engineer": "data engineer",
    "data scientist": "data scientist",
    "backend developer": "backend developer",
    "backend engineer": "backend developer",
    "frontend developer": "frontend developer",
    "fullstack developer": "fullstack developer",
    "full-stack developer": "fullstack developer",
}

def normalize_job_title(text: str) -> str:
    text = normalize_empty_value(text)
    if text is None:
        return ""
    
    text = normalize_unicode(text).strip()
    
    # clean nhẹ trước
    text = clean_text_light(text)
    
    # bỏ một số pattern nhiễu
    for pattern in JOB_TITLE_REMOVE_PATTERNS:
        text = re.sub(pattern, "", text, flags=re.IGNORECASE).strip()
    
    # chuẩn hóa khoảng trắng
    text = re.sub(r"\s+", " ", text).strip()
    
    # canonical mapping
    text_lower = text.lower()
    text_norm = JOB_TITLE_SYNONYM_MAP.get(text_lower, text_lower)
    
    return text_norm

In [17]:
df_clean["job_title_norm"] = df_clean["job_title_raw"].apply(normalize_job_title)

display(
    df_clean[
        ["job_title_raw", "job_title_clean_light", "job_title_norm"]
    ].head(10)
)

,job_title_raw,job_title_clean_light,job_title_norm
0,Junior FP&A Analyst - Hồ Chí Minh,Junior FP A Analyst - Hồ Chí Minh,junior fp a analyst
1,Data Governance Specialist,Data Governance Specialist,data governance specialist
2,Project Manager Dự Án AI HUB,Project Manager Dự Án AI HUB,project manager dự án ai hub
3,AI Engineer,AI Engineer,ai engineer
4,AI Engineer,AI Engineer,ai engineer
5,Data Analyst,Data Analyst,data analyst
6,Data Engineer,Data Engineer,data engineer
7,Fresher Data Engineer,Fresher Data Engineer,fresher data engineer
8,Data Analyst Teamleader (Collection Analytics),Data Analyst Teamleader (Collection Analytics),data analyst teamleader
9,Data Analyst Teamleader (Collection Analytics)...,Data Analyst Teamleader (Collection Analytics)...,data analyst teamleader - thu nhập 50 triệu tạ...


In [18]:
# DANH SÁCH THÀNH PHỐ / TỈNH
VIETNAM_CITIES = {
    # TP. Hà Nội
    "hà nội", "ha noi", "tp hà nội", "tp ha noi",
    # TP. Hồ Chí Minh
    "hồ chí minh", "ho chi minh", "tp hồ chí minh", "tp ho chi minh",
    # TP. Hải Phòng
    "hải phòng", "hai phong", "tp hải phòng", "tp hai phong",
    # TP. Huế
    "Huế", "hue", "tp huế", "tp hue",
    # TP. Đà Nẵng
    "đà nẵng", "da nang", "tp đà nẵng", "tp da nang",
    # TP. Cần Thơ
    "cần thơ", "can tho", "tp cần thơ", "tp can tho",
    
    "an giang", "an giang",
    "bắc ninh", "bac ninh",
    "cà mau", "ca mau",
    "cao bằng", "cao bang",
    "đắk lắk", "dak lak",
    "điện biên", "dien bien",
    "đồng nai", "dong nai",
    "đồng tháp", "dong thap",
    "gia lai", "gia lai",
    "hà tĩnh", "ha tinh",
    "hưng yên", "hung yen",
    "khánh hòa", "khanh hoa",
    "lai châu", "lai chau",
    "lâm đồng", "lam dong",
    "lạng sơn", "lang son",
    "lào cai", "lao cai",
    "nghệ an", "nghe an",
    "ninh bình", "ninh binh",
    "phú thọ", "phu tho",
    "quảng ngãi", "quang ngai",
    "quảng ninh", "quang ninh"
    "quảng trị", "quang tri",
    "sơn la", "son la",
    "tây ninh", "tay ninh",
    "thái nguyên", "thai nguyen",
    "thanh hóa", "thanh hoa",
    "tuyên quang", "tuyen quang",
    "vĩnh long", "vinh long",
}

# =========================
# HÀM TẠO clean location
# =========================
def clean_location(location_raw: str, working_addresses_raw: str) -> str:
    if pd.isna(location_raw) or not str(location_raw).strip():
        return ""
    
    loc = str(location_raw).strip()
    
    # Kiểm tra có nhiều địa điểm không
    multi_keywords = {"và", "&", "or", ",", "với", "cùng", "và các"}
    has_multiple = any(kw in loc.lower() for kw in multi_keywords)
    
    # Trường hợp chỉ 1 địa điểm → trả về nguyên bản
    if not has_multiple:
        return loc
    
    # Trường hợp nhiều địa điểm → trích xuất tất cả từ working_addresses_raw
    if pd.isna(working_addresses_raw) or not str(working_addresses_raw).strip():
        return loc  # fallback
    
    addr = str(working_addresses_raw).lower()   # so sánh không phân biệt hoa thường
    
    found_cities = []
    for city in VIETNAM_CITIES:
        if city in addr:
            # Lấy phiên bản có dấu đẹp để hiển thị
            found_cities.append(city)
    
    if found_cities:
        # Loại bỏ trùng lặp và sắp xếp theo thứ tự xuất hiện trong danh sách gốc
        seen = set()
        unique_cities = []
        for city in found_cities:
            if city not in seen:
                seen.add(city)
                unique_cities.append(city)
        
        return ", ".join(unique_cities)
    
    # Nếu không tìm thấy thành phố nào trong danh sách
    return loc

df_clean["location_clean"] = df_clean.apply(
    lambda row: clean_location(
        row.get("location_raw", ""), 
        row.get("working_addresses_raw", "") or row.get("working_addresses_clean", "")
    ), 
    axis=1
)

In [19]:
def clean_working_addresses_solution_3(raw_text, vietnam_cities):
    DISTRICT_PATTERN = re.compile(
        r'(?i)\b(quận|huyện|thị xã|thành phố)\s+[^\n,;()]+'
    )

    FLOOR_PATTERN = re.compile(
        r'(?i)\b(?:tầng|lầu|floor|fl)\s*[a-z0-9,\-–/ ]+'
    )

    def normalize_empty_value(val):
        if pd.isna(val):
            return None
        val_str = str(val).strip()
        if not val_str or val_str.lower() in {"nan", "none", "null"}:
            return None
        return val_str

    def normalize_unicode(text):
        if text is None:
            return ""
        return unicodedata.normalize("NFC", str(text))

    def clean_address_text(text):
        text = normalize_empty_value(text)
        if text is None:
            return ""
        text = normalize_unicode(text)
        text = text.replace("\\n", "\n")
        text = re.sub(r"[ \t]+", " ", text)
        text = re.sub(r"\s*\n\s*", "\n", text)
        return text.strip()

    def remove_leading_note(text):
        text = clean_address_text(text)
        if not text:
            return ""
        return re.sub(r'^\s*\([^)]*\)\s*', '', text).strip()

    def detect_city_from_text(text):
        text = clean_address_text(text)
        if not text:
            return None
        text_lower = text.lower()
        for city in vietnam_cities:
            city_norm = normalize_empty_value(city)
            if city_norm and city_norm.lower() in text_lower:
                return city_norm
        return None

    def extract_district_note(text):
        text = clean_address_text(text)
        if not text:
            return None
        matches = re.findall(r'\(([^)]*)\)', text)
        matches = [m.strip() for m in matches if normalize_empty_value(m)]
        return "; ".join(matches) if matches else None

    def remove_parentheses_content(text):
        text = clean_address_text(text)
        if not text:
            return ""
        return re.sub(r'\([^)]*\)', '', text).strip()

    def classify_chunk(chunk):
        chunk_lower = chunk.lower().strip()

        if re.search(r'\b(phường|xã|thị trấn)\b', chunk_lower):
            return "ward"

        if re.search(r'\b(tầng|lầu|floor|fl)\b', chunk_lower):
            return "floor_or_building"

        if re.search(r'\b(?:tòa|toà|tower|building|plaza|center|centre|complex|landmark|office)\b', chunk_lower):
            return "building"

        if re.search(r'\b(?:số\s*)?\d+[a-z0-9/\-]*\s+', chunk_lower):
            return "street"

        return "other"

    def parse_floor_and_building_from_chunk(chunk):
        floor_match = FLOOR_PATTERN.search(chunk)
        floor = floor_match.group(0).strip() if floor_match else None

        building = chunk
        if floor:
            building = FLOOR_PATTERN.sub("", building).strip(" ,-")
        if not building:
            building = None

        return floor, building

    def empty_result():
        return {
            "city": None,
            "ward": None,
            "district": None,
            "district_note": None,
            "floor": None,
            "building": None,
            "street_address": None,
            "location_detail": None,
            "full_parsed": None,
            "parse_method": "solution_3"
        }

    raw_text = normalize_empty_value(raw_text)
    if raw_text is None:
        return empty_result()

    text = clean_address_text(raw_text)
    text = remove_leading_note(text)
    text = re.sub(r'^\-\s*', '', text).strip()

    if not text:
        return empty_result()

    city = None
    detail = text

    if ":" in text:
        left, right = text.split(":", 1)
        detail = right.strip()
        city = detect_city_from_text(left.strip())

    if not city:
        city = detect_city_from_text(text)

    district_note = extract_district_note(detail)
    detail_no_note = remove_parentheses_content(detail)

    district_match = DISTRICT_PATTERN.search(detail_no_note)
    district = district_match.group(0).strip() if district_match else None

    chunks = [c.strip() for c in detail_no_note.split(",") if c.strip()]

    ward = None
    floor = None
    building = None
    street_address = None
    other_chunks = []

    for chunk in chunks:
        label = classify_chunk(chunk)

        if label == "ward" and ward is None:
            ward = chunk

        elif label == "floor_or_building":
            chunk_floor, chunk_building = parse_floor_and_building_from_chunk(chunk)
            if floor is None and chunk_floor:
                floor = chunk_floor
            if chunk_building:
                building = f"{building}, {chunk_building}" if building else chunk_building

        elif label == "building":
            building = f"{building}, {chunk}" if building else chunk

        elif label == "street" and street_address is None:
            street_address = chunk

        else:
            other_chunks.append(chunk)

    location_detail_parts = []
    if floor:
        location_detail_parts.append(floor)
    if building:
        location_detail_parts.append(building)
    if street_address:
        location_detail_parts.append(street_address)
    if other_chunks:
        location_detail_parts.extend(other_chunks)

    location_detail = ", ".join(location_detail_parts) if location_detail_parts else None

    full_parts = [street_address or building or location_detail, ward, city]
    full_parsed = ", ".join([p for p in full_parts if p]) if any(full_parts) else None

    return {
        "city": city,
        "ward": ward,
        "district": district,
        "district_note": district_note,
        "floor": floor,
        "building": building,
        "street_address": street_address,
        "location_detail": location_detail,
        "full_parsed": full_parsed,
        "parse_method": "solution_3"
    }

In [20]:
# =========================
# NORMALIZE WORKING ADDRESS + LOCATION
# =========================

LOCATION_CANONICAL_MAP = {
    "hcm": "hồ chí minh",
    "tp hcm": "hồ chí minh",
    "tp. hcm": "hồ chí minh",
    "tphcm": "hồ chí minh",
    "ho chi minh": "hồ chí minh",
    "tp ho chi minh": "hồ chí minh",
    "tp. ho chi minh": "hồ chí minh",
    "ha noi": "hà nội",
    "tp ha noi": "hà nội",
    "tp. ha noi": "hà nội",
    "da nang": "đà nẵng",
    "tp da nang": "đà nẵng",
    "tp. da nang": "đà nẵng",
}

def canonicalize_location_name(text: str) -> str:
    text = normalize_empty_value(text)
    if text is None:
        return ""
    
    text = clean_text_strict(text)
    text = LOCATION_CANONICAL_MAP.get(text, text)
    return text.strip()

def clean_working_addresses_norm(raw_text: str) -> str:
    """
    Ưu tiên tận dụng clean_working_addresses_solution_3 nếu notebook đã có.
    Nếu lỗi thì fallback về clean_text_light.
    """
    raw_text = normalize_empty_value(raw_text)
    if raw_text is None:
        return ""
    
    try:
        cleaned = clean_working_addresses_solution_3(raw_text, VIETNAM_CITIES)
        
        # Nếu hàm cũ trả về dict/list thì convert về text
        if isinstance(cleaned, list):
            cleaned = " | ".join([str(x) for x in cleaned if x])
        elif isinstance(cleaned, dict):
            cleaned = " | ".join([f"{k}: {v}" for k, v in cleaned.items() if v])
        elif cleaned is None:
            cleaned = ""
        
        cleaned = clean_text_light(cleaned)
        return cleaned
    except:
        return clean_text_light(raw_text)

def normalize_location(location_raw: str, working_addresses_raw: str) -> str:
    """
    Tận dụng logic clean_location cũ nếu có.
    Sau đó canonicalize về tên địa điểm chuẩn.
    """
    location_raw = normalize_empty_value(location_raw)
    working_addresses_raw = normalize_empty_value(working_addresses_raw)

    if location_raw is None and working_addresses_raw is None:
        return ""
    
    try:
        loc = clean_location(location_raw, working_addresses_raw)
    except:
        loc = first_non_empty(location_raw, working_addresses_raw)
    
    if loc is None:
        return ""
    
    loc = normalize_unicode(str(loc))
    loc = re.sub(r"\s+", " ", loc).strip()
    
    # tách nhiều địa điểm nếu có
    parts = re.split(r"[;,/|]| và ", loc, flags=re.IGNORECASE)
    parts = [canonicalize_location_name(p) for p in parts if normalize_empty_value(p)]
    parts = [p for p in parts if p]
    
    # unique nhưng giữ thứ tự
    seen = set()
    final_parts = []
    for p in parts:
        if p not in seen:
            seen.add(p)
            final_parts.append(p)
    
    return " | ".join(final_parts)

In [21]:
df_clean["working_addresses_norm"] = df_clean["working_addresses_raw"].apply(clean_working_addresses_norm)

df_clean["location_norm"] = df_clean.apply(
    lambda row: normalize_location(row["location_raw"], row["working_addresses_raw"]),
    axis=1
)

display(
    df_clean[
        ["location_raw", "working_addresses_raw", "working_addresses_norm", "location_norm"]
    ].head(10)
)

,location_raw,working_addresses_raw,working_addresses_norm,location_norm
0,Hồ Chí Minh,(đã được cập nhật theo Danh mục Hành chính mới...,city: hồ chí minh ward: Phường Tân Sơn Hòa dis...,hồ chí minh
1,Hà Nội,(đã được cập nhật theo Danh mục Hành chính mới...,city: hà nội ward: Phường Thanh Xuân\nHà Nội: ...,hà nội
2,Hà Nội,(đã được cập nhật theo Danh mục Hành chính mới...,city: hà nội ward: Phường Thanh Xuân\nHà Nội: ...,hà nội
3,Hà Nội,(đã được cập nhật theo Danh mục Hành chính mới...,city: hà nội ward: Phường Cầu Giấy district_no...,hà nội
4,Hà Nội,(đã được cập nhật theo Danh mục Hành chính mới...,city: hà nội ward: Phường Yên Hòa district_not...,hà nội
5,Hà Nội,(đã được cập nhật theo Danh mục Hành chính mới...,city: hà nội ward: Phường Yên Hòa district_not...,hà nội
6,Hà Nội,(đã được cập nhật theo Danh mục Hành chính mới...,city: hà nội ward: Phường Yên Hòa district_not...,hà nội
7,Hà Nội,(đã được cập nhật theo Danh mục Hành chính mới...,city: hà nội ward: Xã Hòa Lạc - Hà Nội: FPT To...,hà nội
8,Hồ Chí Minh,(đã được cập nhật theo Danh mục Hành chính mới...,city: hồ chí minh ward: Phường Bảy Hiền distri...,hồ chí minh
9,Hồ Chí Minh,(đã được cập nhật theo Danh mục Hành chính mới...,city: hồ chí minh ward: Phường Bảy Hiền distri...,hồ chí minh


In [22]:
def clean_salary(raw: str) -> str:
    text = normalize_empty_value(raw)
    if text is None:
        return ""
    
    text = normalize_unicode(text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    
    # Chuẩn hóa khoảng trắng quanh dấu gạch nối
    text = re.sub(r"\s*-\s*", " - ", text)
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [23]:
# =========================
# NORMALIZE SALARY
# =========================

def parse_salary_range(text: str):
    text = normalize_empty_value(text)
    if text is None:
        return {
            "salary_clean": "",
            "salary_min": None,
            "salary_max": None,
            "salary_currency": None,
            "salary_period": None,
            "salary_type": None,
            "salary_negotiable": 0
        }
    
    text_clean = clean_salary(text)
    text_lower = text_clean.lower()
    
    # cờ thỏa thuận
    negotiable_keywords = ["thỏa thuận", "thu nhập cạnh tranh", "cạnh tranh", "deal lương", "negotiable"]
    is_negotiable = int(any(k in text_lower for k in negotiable_keywords))
    
    # xác định đơn vị
    currency = "VND"
    if "usd" in text_lower or "$" in text_lower:
        currency = "USD"
    
    period = "month"
    if any(x in text_lower for x in ["năm", "/năm", "year"]):
        period = "year"
    elif any(x in text_lower for x in ["giờ", "/giờ", "hour"]):
        period = "hour"
    
    # chuẩn hóa số
    nums = re.findall(r"\d+(?:[\.,]\d+)?", text_lower)
    nums = [float(x.replace(",", ".")) for x in nums] if nums else []
    
    salary_min, salary_max = None, None
    
    if len(nums) >= 2:
        salary_min, salary_max = nums[0], nums[1]
    elif len(nums) == 1:
        salary_min = nums[0]
        salary_max = nums[0]
    
    salary_type = None
    if salary_min is not None and salary_max is not None:
        salary_type = "range" if salary_min != salary_max else "fixed"
    elif is_negotiable:
        salary_type = "negotiable"
    
    return {
        "salary_clean": text_clean,
        "salary_min": salary_min,
        "salary_max": salary_max,
        "salary_currency": currency,
        "salary_period": period,
        "salary_type": salary_type,
        "salary_negotiable": is_negotiable
    }

In [24]:
salary_parsed = df_clean["salary_raw"].apply(parse_salary_range)

df_clean["salary_clean"] = salary_parsed.apply(lambda x: x["salary_clean"])
df_clean["salary_min"] = salary_parsed.apply(lambda x: x["salary_min"])
df_clean["salary_max"] = salary_parsed.apply(lambda x: x["salary_max"])
df_clean["salary_currency"] = salary_parsed.apply(lambda x: x["salary_currency"])
df_clean["salary_period"] = salary_parsed.apply(lambda x: x["salary_period"])
df_clean["salary_type"] = salary_parsed.apply(lambda x: x["salary_type"])
df_clean["salary_negotiable"] = salary_parsed.apply(lambda x: x["salary_negotiable"])

display(
    df_clean[
        ["salary_raw", "salary_clean", "salary_min", "salary_max", "salary_currency", "salary_period", "salary_type", "salary_negotiable"]
    ].head(10)
)

,salary_raw,salary_clean,salary_min,salary_max,salary_currency,salary_period,salary_type,salary_negotiable
0,12 - 20 triệu,12 - 20 triệu,12.0,20.0,VND,month,range,0
1,20 - 30 triệu,20 - 30 triệu,20.0,30.0,VND,month,range,0
2,30 - 35 triệu,30 - 35 triệu,30.0,35.0,VND,month,range,0
3,Thoả thuận,Thoả thuận,NaN,NaN,VND,month,NaN,0
4,Thoả thuận,Thoả thuận,NaN,NaN,VND,month,NaN,0
5,Thoả thuận,Thoả thuận,NaN,NaN,VND,month,NaN,0
6,Thoả thuận,Thoả thuận,NaN,NaN,VND,month,NaN,0
7,6 - 9 triệu,6 - 9 triệu,6.0,9.0,VND,month,range,0
8,Thoả thuận,Thoả thuận,NaN,NaN,VND,month,NaN,0
9,Thoả thuận,Thoả thuận,NaN,NaN,VND,month,NaN,0


In [25]:
def clean_experience(text: str):
    text = normalize_empty_value(text)
    if text is None:
        return None
    
    text = normalize_unicode(text).lower().strip()

    # Không yêu cầu
    if "không yêu cầu" in text:
        return 0

    # Dưới 1 năm
    if "dưới 1 năm" in text:
        return 0

    # Lấy tất cả số
    nums = [int(x) for x in re.findall(r"\d+", text)]
    if not nums:
        return None

    # Nếu là khoảng thì lấy min
    return nums[0]

In [26]:
# =========================
# NORMALIZE EXPERIENCE
# =========================

def parse_experience_range(text: str):
    raw = normalize_empty_value(text)
    if raw is None:
        return {
            "experience_clean": "",
            "experience_min_years": None,
            "experience_max_years": None
        }
    
    text_clean = clean_text_strict(raw)
    
    if "không yêu cầu" in text_clean or "chưa có kinh nghiệm" in text_clean:
        return {
            "experience_clean": text_clean,
            "experience_min_years": 0,
            "experience_max_years": 0
        }
    
    if "dưới 1 năm" in text_clean:
        return {
            "experience_clean": text_clean,
            "experience_min_years": 0,
            "experience_max_years": 1
        }
    
    nums = [int(x) for x in re.findall(r"\d+", text_clean)]
    
    if not nums:
        return {
            "experience_clean": text_clean,
            "experience_min_years": clean_experience(raw),  # fallback logic cũ
            "experience_max_years": clean_experience(raw)
        }
    
    if len(nums) >= 2:
        return {
            "experience_clean": text_clean,
            "experience_min_years": nums[0],
            "experience_max_years": nums[1]
        }
    
    val = nums[0]
    
    if any(k in text_clean for k in ["trên", "hơn", "tối thiểu", "at least", "minimum"]):
        return {
            "experience_clean": text_clean,
            "experience_min_years": val,
            "experience_max_years": None
        }
    
    return {
        "experience_clean": text_clean,
        "experience_min_years": val,
        "experience_max_years": val
    }

In [27]:
exp_parsed = df_clean["experience_raw"].apply(parse_experience_range)

df_clean["experience_clean"] = exp_parsed.apply(lambda x: x["experience_clean"])
df_clean["experience_min_years"] = exp_parsed.apply(lambda x: x["experience_min_years"])
df_clean["experience_max_years"] = exp_parsed.apply(lambda x: x["experience_max_years"])

display(
    df_clean[
        ["experience_raw", "experience_clean", "experience_min_years", "experience_max_years"]
    ].head(10)
)

,experience_raw,experience_clean,experience_min_years,experience_max_years
0,2 năm,2 năm,2,2.0
1,2 năm,2 năm,2,2.0
2,2 năm,2 năm,2,2.0
3,3 năm,3 năm,3,3.0
4,2 năm,2 năm,2,2.0
5,3 năm,3 năm,3,3.0
6,3 năm,3 năm,3,3.0
7,Không yêu cầu,không yêu cầu,0,0.0
8,1 năm,1 năm,1,1.0
9,1 năm,1 năm,1,1.0


In [28]:
# =========================
# NORMALIZE EDUCATION LEVEL
# =========================

EDUCATION_MAP = {
    "trung học": "high_school",
    "thpt": "high_school",
    "cao đẳng": "college",
    "đại học": "bachelor",
    "cử nhân": "bachelor",
    "bachelor": "bachelor",
    "thạc sĩ": "master",
    "master": "master",
    "tiến sĩ": "phd",
    "phd": "phd",
}

def normalize_education_level(text: str) -> str:
    text = normalize_empty_value(text)
    if text is None:
        return ""
    
    text = clean_text_strict(text)
    
    for key, value in EDUCATION_MAP.items():
        if key in text:
            return value
    
    return text

In [29]:
df_clean["education_level_norm"] = df_clean["education_level_raw"].apply(normalize_education_level)

display(
    df_clean[
        ["education_level_raw", "education_level_norm"]
    ].head(10)
)

,education_level_raw,education_level_norm
0,Đại Học trở lên,bachelor
1,Đại Học trở lên,bachelor
2,Đại Học trở lên,bachelor
3,Đại Học trở lên,bachelor
4,Đại Học trở lên,bachelor
5,Đại Học trở lên,bachelor
6,Đại Học trở lên,bachelor
7,Đại Học trở lên,bachelor
8,Đại Học trở lên,bachelor
9,Đại Học trở lên,bachelor


In [30]:
# =========================
# NORMALIZE EMPLOYMENT TYPE
# =========================

EMPLOYMENT_TYPE_MAP = {
    "toàn thời gian": "full_time",
    "full time": "full_time",
    "full-time": "full_time",
    "bán thời gian": "part_time",
    "part time": "part_time",
    "part-time": "part_time",
    "thực tập": "internship",
    "intern": "internship",
    "internship": "internship",
    "freelance": "freelance",
    "cộng tác viên": "freelance",
    "hợp đồng": "contract",
    "contract": "contract",
    "remote": "remote",
    "hybrid": "hybrid",
}

def normalize_employment_type(text: str) -> str:
    text = normalize_empty_value(text)
    if text is None:
        return ""
    
    text = clean_text_strict(text)
    
    for key, value in EMPLOYMENT_TYPE_MAP.items():
        if key in text:
            return value
    
    return text

In [31]:
df_clean["employment_type_norm"] = df_clean["employment_type_raw"].apply(normalize_employment_type)

display(
    df_clean[
        ["employment_type_raw", "employment_type_norm"]
    ].head(10)
)

,employment_type_raw,employment_type_norm
0,Toàn thời gian,full_time
1,Toàn thời gian,full_time
2,Toàn thời gian,full_time
3,Toàn thời gian,full_time
4,Toàn thời gian,full_time
5,Toàn thời gian,full_time
6,Toàn thời gian,full_time
7,Toàn thời gian,full_time
8,Toàn thời gian,full_time
9,Toàn thời gian,full_time


In [32]:
# =========================
# NORMALIZE JOB LEVEL
# =========================

JOB_LEVEL_MAP = {
    "intern": "intern",
    "thực tập": "intern",
    "fresher": "fresher",
    "junior": "junior",
    "nhân viên": "junior",
    "chuyên viên": "junior",
    "mid": "middle",
    "middle": "middle",
    "senior": "senior",
    "trưởng nhóm": "lead",
    "team lead": "lead",
    "leader": "lead",
    "lead": "lead",
    "manager": "manager",
    "quản lý": "manager",
    "head": "manager",
    "director": "director",
}

def normalize_job_level(text: str, title_text: str = None) -> str:
    merged = " ".join([
        clean_text_strict(text) if normalize_empty_value(text) else "",
        clean_text_strict(title_text) if normalize_empty_value(title_text) else ""
    ]).strip()
    
    if not merged:
        return ""
    
    for key, value in JOB_LEVEL_MAP.items():
        if key in merged:
            return value
    
    return ""

In [33]:
df_clean["job_level_norm"] = df_clean.apply(
    lambda row: normalize_job_level(row["job_level_raw"], row["job_title_raw"]),
    axis=1
)

display(
    df_clean[
        ["job_level_raw", "job_title_raw", "job_level_norm"]
    ].head(10)
)

,job_level_raw,job_title_raw,job_level_norm
0,Nhân viên,Junior FP&A Analyst - Hồ Chí Minh,junior
1,Nhân viên,Data Governance Specialist,junior
2,Nhân viên,Project Manager Dự Án AI HUB,junior
3,Nhân viên,AI Engineer,junior
4,Nhân viên,AI Engineer,junior
5,Nhân viên,Data Analyst,junior
6,Nhân viên,Data Engineer,junior
7,Trưởng nhóm,Fresher Data Engineer,fresher
8,Trưởng nhóm,Data Analyst Teamleader (Collection Analytics),lead
9,Trưởng nhóm,Data Analyst Teamleader (Collection Analytics)...,lead


In [34]:
metadata_preview_cols = [
    "job_title_raw", "job_title_norm",
    "location_raw", "working_addresses_norm", "location_norm",
    "salary_raw", "salary_min", "salary_max", "salary_type",
    "experience_raw", "experience_min_years", "experience_max_years",
    "education_level_raw", "education_level_norm",
    "employment_type_raw", "employment_type_norm",
    "job_level_raw", "job_level_norm"
]

metadata_preview_cols = [c for c in metadata_preview_cols if c in df_clean.columns]

display(df_clean[metadata_preview_cols].head(10))

,job_title_raw,job_title_norm,location_raw,working_addresses_norm,location_norm,salary_raw,salary_min,salary_max,salary_type,experience_raw,experience_min_years,experience_max_years,education_level_raw,education_level_norm,employment_type_raw,employment_type_norm,job_level_raw,job_level_norm
0,Junior FP&A Analyst - Hồ Chí Minh,junior fp a analyst,Hồ Chí Minh,city: hồ chí minh ward: Phường Tân Sơn Hòa dis...,hồ chí minh,12 - 20 triệu,12.0,20.0,range,2 năm,2,2.0,Đại Học trở lên,bachelor,Toàn thời gian,full_time,Nhân viên,junior
1,Data Governance Specialist,data governance specialist,Hà Nội,city: hà nội ward: Phường Thanh Xuân\nHà Nội: ...,hà nội,20 - 30 triệu,20.0,30.0,range,2 năm,2,2.0,Đại Học trở lên,bachelor,Toàn thời gian,full_time,Nhân viên,junior
2,Project Manager Dự Án AI HUB,project manager dự án ai hub,Hà Nội,city: hà nội ward: Phường Thanh Xuân\nHà Nội: ...,hà nội,30 - 35 triệu,30.0,35.0,range,2 năm,2,2.0,Đại Học trở lên,bachelor,Toàn thời gian,full_time,Nhân viên,junior
3,AI Engineer,ai engineer,Hà Nội,city: hà nội ward: Phường Cầu Giấy district_no...,hà nội,Thoả thuận,NaN,NaN,NaN,3 năm,3,3.0,Đại Học trở lên,bachelor,Toàn thời gian,full_time,Nhân viên,junior
4,AI Engineer,ai engineer,Hà Nội,city: hà nội ward: Phường Yên Hòa district_not...,hà nội,Thoả thuận,NaN,NaN,NaN,2 năm,2,2.0,Đại Học trở lên,bachelor,Toàn thời gian,full_time,Nhân viên,junior
5,Data Analyst,data analyst,Hà Nội,city: hà nội ward: Phường Yên Hòa district_not...,hà nội,Thoả thuận,NaN,NaN,NaN,3 năm,3,3.0,Đại Học trở lên,bachelor,Toàn thời gian,full_time,Nhân viên,junior
6,Data Engineer,data engineer,Hà Nội,city: hà nội ward: Phường Yên Hòa district_not...,hà nội,Thoả thuận,NaN,NaN,NaN,3 năm,3,3.0,Đại Học trở lên,bachelor,Toàn thời gian,full_time,Nhân viên,junior
7,Fresher Data Engineer,fresher data engineer,Hà Nội,city: hà nội ward: Xã Hòa Lạc - Hà Nội: FPT To...,hà nội,6 - 9 triệu,6.0,9.0,range,Không yêu cầu,0,0.0,Đại Học trở lên,bachelor,Toàn thời gian,full_time,Trưởng nhóm,fresher
8,Data Analyst Teamleader (Collection Analytics),data analyst teamleader,Hồ Chí Minh,city: hồ chí minh ward: Phường Bảy Hiền distri...,hồ chí minh,Thoả thuận,NaN,NaN,NaN,1 năm,1,1.0,Đại Học trở lên,bachelor,Toàn thời gian,full_time,Trưởng nhóm,lead
9,Data Analyst Teamleader (Collection Analytics)...,data analyst teamleader - thu nhập 50 triệu tạ...,Hồ Chí Minh,city: hồ chí minh ward: Phường Bảy Hiền distri...,hồ chí minh,Thoả thuận,NaN,NaN,NaN,1 năm,1,1.0,Đại Học trở lên,bachelor,Toàn thời gian,full_time,Trưởng nhóm,lead


In [35]:
normalize_check_cols = [
    "job_title_norm",
    "location_norm",
    "salary_min",
    "experience_min_years",
    "education_level_norm",
    "employment_type_norm",
    "job_level_norm"
]

report = {}
for col in normalize_check_cols:
    if col in df_clean.columns:
        if df_clean[col].dtype == "O":
            empty_ratio = (df_clean[col].fillna("").astype(str).str.strip() == "").mean()
            report[col] = {
                "missing_or_empty_ratio": round(empty_ratio, 4),
                "nunique": df_clean[col].nunique(dropna=True)
            }
        else:
            report[col] = {
                "missing_or_empty_ratio": round(df_clean[col].isna().mean(), 4),
                "nunique": df_clean[col].nunique(dropna=True)
            }

display(pd.DataFrame(report).T)

,missing_or_empty_ratio,nunique
job_title_norm,0.0000,228.0
location_norm,0.0000,14.0
salary_min,0.4923,38.0
experience_min_years,0.0000,6.0
education_level_norm,0.0000,5.0
employment_type_norm,0.0000,5.0
job_level_norm,0.0000,8.0


In [36]:
# =========================
# NORMALIZE TAGS
# =========================

def normalize_tags(text: str):
    text = normalize_empty_value(text)
    if text is None:
        return []
    
    text = clean_text_strict(text)
    
    # split theo nhiều dạng phân tách
    parts = re.split(r"[;,/|]", text)
    
    parts = [p.strip() for p in parts if p.strip()]
    
    # unique giữ thứ tự
    seen = set()
    final = []
    for p in parts:
        if p not in seen:
            seen.add(p)
            final.append(p)
    
    return final

In [37]:
df_clean["tags_list"] = df_clean["tags_raw"].apply(normalize_tags)

display(
    df_clean[
        ["tags_raw", "tags_list"]
    ].head(10)
)

,tags_raw,tags_list
0,Chuyên môn Data Analyst; Tài chính; Kế toán,[chuyên môn data analyst tài chính kế toán]
1,Chuyên môn Data Analyst,[chuyên môn data analyst]
2,Chuyên môn IT Project Manager,[chuyên môn it project manager]
3,Chuyên môn AI Engineer; IT - Phần mềm,[chuyên môn ai engineer it - phần mềm]
4,Chuyên môn AI Engineer,[chuyên môn ai engineer]
5,Chuyên môn Data Analyst,[chuyên môn data analyst]
6,Chuyên môn Data Engineer,[chuyên môn data engineer]
7,Chuyên môn Data Engineer; IT - Phần mềm,[chuyên môn data engineer it - phần mềm]
8,Chuyên môn Data Analyst; Tài chính; Ngân hàng;...,[chuyên môn data analyst tài chính ngân hàng i...
9,Chuyên môn Data Analyst; Tài chính; Ngân hàng,[chuyên môn data analyst tài chính ngân hàng]


In [38]:
# =========================
# SKILL DICTIONARY
# =========================

SKILL_DICT = {
    # programming
    "python": ["python"],
    "r": [" r ", " r,", " r."],
    "java": ["java"],
    "scala": ["scala"],
    "c++": ["c++"],
    "c#": ["c#"],
    
    # database
    "sql": ["sql", "mysql", "postgres", "oracle", "sql server"],
    "mongodb": ["mongodb"],
    
    # data stack
    "etl": ["etl", "elt"],
    "data warehouse": ["data warehouse", "dwh"],
    "data lake": ["data lake"],
    "airflow": ["airflow"],
    "dbt": ["dbt"],
    "spark": ["spark"],
    "hadoop": ["hadoop"],
    
    # BI / visualization
    "power bi": ["power bi", "powerbi", "pbi"],
    "tableau": ["tableau"],
    "looker": ["looker"],
    
    # analytics
    "statistics": ["statistics", "thống kê"],
    "a/b testing": ["a/b", "ab testing"],
    "forecasting": ["forecast"],
    
    # ml/ai
    "machine learning": ["machine learning", "ml"],
    "deep learning": ["deep learning", "dl"],
    
    # cloud
    "aws": ["aws"],
    "gcp": ["gcp"],
    "azure": ["azure"],
    
    # tools
    "excel": ["excel"],
    "powerpoint": ["powerpoint"],
    
    # governance
    "data governance": ["data governance"],
    "data quality": ["data quality"],
    "data lineage": ["data lineage"]
}

In [39]:
# =========================
# EXTRACT SKILLS
# =========================

def extract_skills_from_text(text: str):
    text = normalize_empty_value(text)
    if text is None:
        return []
    
    text = clean_text_strict(text)
    
    found_skills = []
    
    for skill, patterns in SKILL_DICT.items():
        for p in patterns:
            if p in text:
                found_skills.append(skill)
                break
    
    return found_skills

In [40]:
# extract riêng từng nguồn
df_clean["skills_from_tags"] = df_clean["tags_raw"].apply(extract_skills_from_text)

df_clean["skills_from_title"] = df_clean["job_title_clean_light"].apply(extract_skills_from_text)

df_clean["skills_from_requirements"] = df_clean["requirements_clean_light"].apply(extract_skills_from_text)

df_clean["skills_from_description"] = df_clean["description_clean_light"].apply(extract_skills_from_text)

In [41]:
def merge_skill_lists(*lists):
    merged = []
    seen = set()
    
    for lst in lists:
        if not lst:
            continue
        for item in lst:
            if item not in seen:
                seen.add(item)
                merged.append(item)
    
    return merged

In [42]:
df_clean["skills_extracted"] = df_clean.apply(
    lambda row: merge_skill_lists(
        row["skills_from_tags"],
        row["skills_from_title"],
        row["skills_from_requirements"],
        row["skills_from_description"]
    ),
    axis=1
)

display(
    df_clean[
        [
            "job_title_raw",
            "tags_raw",
            "skills_from_tags",
            "skills_from_title",
            "skills_from_requirements",
            "skills_extracted"
        ]
    ].head(10)
)

,job_title_raw,tags_raw,skills_from_tags,skills_from_title,skills_from_requirements,skills_extracted
0,Junior FP&A Analyst - Hồ Chí Minh,Chuyên môn Data Analyst; Tài chính; Kế toán,[],[],[deep learning],[deep learning]
1,Data Governance Specialist,Chuyên môn Data Analyst,[],[data governance],"[sql, etl, data warehouse, data governance]","[data governance, sql, etl, data warehouse, da..."
2,Project Manager Dự Án AI HUB,Chuyên môn IT Project Manager,[],[],[],[]
3,AI Engineer,Chuyên môn AI Engineer; IT - Phần mềm,[],[],"[python, machine learning]","[python, machine learning]"
4,AI Engineer,Chuyên môn AI Engineer,[],[],[],[]
5,Data Analyst,Chuyên môn Data Analyst,[],[],"[sql, etl, data warehouse, machine learning]","[sql, etl, data warehouse, machine learning]"
6,Data Engineer,Chuyên môn Data Engineer,[],[],"[sql, etl]","[sql, etl]"
7,Fresher Data Engineer,Chuyên môn Data Engineer; IT - Phần mềm,[],[],"[python, sql, etl, spark, machine learning]","[python, sql, etl, spark, machine learning, po..."
8,Data Analyst Teamleader (Collection Analytics),Chuyên môn Data Analyst; Tài chính; Ngân hàng;...,[],[machine learning],"[python, sql]","[machine learning, python, sql, data warehouse]"
9,Data Analyst Teamleader (Collection Analytics)...,Chuyên môn Data Analyst; Tài chính; Ngân hàng,[],[machine learning],"[python, sql]","[machine learning, python, sql, data warehouse]"


In [43]:
df_clean["skills_text"] = df_clean["skills_extracted"].apply(
    lambda x: "; ".join(x) if isinstance(x, list) else ""
)

In [44]:
df_clean["num_skills"] = df_clean["skills_extracted"].apply(lambda x: len(x) if isinstance(x, list) else 0)

display(df_clean["num_skills"].describe())

print("Avg skills per job:", round(df_clean["num_skills"].mean(), 2))

count    325.000000
mean       3.827692
std        3.098362
min        0.000000
25%        1.000000
50%        3.000000
75%        6.000000
max       15.000000
Name: num_skills, dtype: float64

Avg skills per job: 3.83


In [45]:
from collections import Counter

all_skills = df_clean["skills_extracted"].explode().dropna().tolist()

skill_counts = Counter(all_skills)

top_skills = pd.DataFrame(skill_counts.most_common(30), columns=["skill", "count"])

display(top_skills)

,skill,count
0,sql,148
1,python,148
2,machine learning,113
3,deep learning,81
4,etl,62
5,power bi,62
6,excel,55
7,statistics,54
8,spark,48
9,data warehouse,47


In [46]:
empty_skill_ratio = (df_clean["num_skills"] == 0).mean()

print(f"Tỷ lệ job không extract được skill: {round(empty_skill_ratio, 4)}")

Tỷ lệ job không extract được skill: 0.1723


In [47]:
# =========================
# BUILD JOB TEXT FOR MATCHING (PhoBERT)
# =========================

def safe_str(val):
    if val is None:
        return ""
    return str(val).strip()

def build_job_text_matching(row):
    parts = []

    # TITLE (quan trọng nhất)
    if row.get("job_title_norm"):
        parts.append(f"[TITLE] {safe_str(row['job_title_norm'])}")

    # SKILLS (rất quan trọng)
    if row.get("skills_text"):
        parts.append(f"[SKILLS] {safe_str(row['skills_text'])}")

    # EXPERIENCE
    exp_min = row.get("experience_min_years")
    exp_max = row.get("experience_max_years")

    if exp_min is not None or exp_max is not None:
        if exp_min is not None and exp_max is not None:
            parts.append(f"[EXPERIENCE] {exp_min}-{exp_max} years")
        elif exp_min is not None:
            parts.append(f"[EXPERIENCE] >= {exp_min} years")

    # EDUCATION
    if row.get("education_level_norm"):
        parts.append(f"[EDUCATION] {safe_str(row['education_level_norm'])}")

    # LOCATION
    if row.get("location_norm"):
        parts.append(f"[LOCATION] {safe_str(row['location_norm'])}")

    # EMPLOYMENT TYPE
    if row.get("employment_type_norm"):
        parts.append(f"[EMPLOYMENT] {safe_str(row['employment_type_norm'])}")

    # JOB LEVEL
    if row.get("job_level_norm"):
        parts.append(f"[LEVEL] {safe_str(row['job_level_norm'])}")

    # DESCRIPTION (rút gọn)
    desc = safe_str(row.get("description_clean_light"))
    if desc:
        desc = desc[:800]  # tránh quá dài
        parts.append(f"[DESCRIPTION] {desc}")

    # REQUIREMENTS (rút gọn)
    req = safe_str(row.get("requirements_clean_light"))
    if req:
        req = req[:800]
        parts.append(f"[REQUIREMENTS] {req}")

    return "\n".join(parts)

In [48]:
df_clean["job_text_matching"] = df_clean.apply(build_job_text_matching, axis=1)

print("[INFO] Đã tạo xong job_text_matching")

[INFO] Đã tạo xong job_text_matching


In [49]:
display(
    df_clean[
        [
            "job_title_raw",
            "job_title_norm",
            "skills_text",
            "job_text_matching"
        ]
    ].head(3)
)

,job_title_raw,job_title_norm,skills_text,job_text_matching
0,Junior FP&A Analyst - Hồ Chí Minh,junior fp a analyst,deep learning,[TITLE] junior fp a analyst\n[SKILLS] deep lea...
1,Data Governance Specialist,data governance specialist,data governance; sql; etl; data warehouse; dat...,[TITLE] data governance specialist\n[SKILLS] d...
2,Project Manager Dự Án AI HUB,project manager dự án ai hub,,[TITLE] project manager dự án ai hub\n[EXPERIE...


In [50]:
df_clean["job_text_len"] = df_clean["job_text_matching"].str.len()

display(df_clean["job_text_len"].describe())

count     325.000000
mean     1510.483077
std       332.820852
min       541.000000
25%      1288.000000
50%      1612.000000
75%      1798.000000
max      1986.000000
Name: job_text_len, dtype: float64

In [51]:
print("=== SHORT TEXT ===")
display(df_clean[df_clean["job_text_len"] < 100][["job_title_raw", "job_text_matching"]].head(3))

print("=== LONG TEXT ===")
display(df_clean[df_clean["job_text_len"] > 2000][["job_title_raw", "job_text_matching"]].head(3))

=== SHORT TEXT ===


,job_title_raw,job_text_matching


=== LONG TEXT ===


,job_title_raw,job_text_matching


# - PHOBERT

In [80]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_NAME = "vinai/phobert-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)

model.to(device)
model.eval()

print("[INFO] PhoBERT loaded on", device)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 18137.04it/s]
RobertaModel LOAD REPORT from: vinai/phobert-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
lm_head.layer_norm.weight       | UNEXPECTED |  | 
lm_head.dense.bias              | UNEXPECTED |  | 
lm_head.decoder.bias            | UNEXPECTED |  | 
lm_head.dense.weight            | UNEXPECTED |  | 
lm_head.decoder.weight          | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
lm_head.bias                    | UNEXPECTED |  | 
lm_head.layer_norm.bias         | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[INFO] PhoBERT loaded on cpu


In [53]:
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]  # (batch, seq_len, hidden)
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

In [54]:
def encode_texts(texts, batch_size=16, max_length=256):
    embeddings = []
    
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )
        
        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded["attention_mask"].to(device)
        
        with torch.no_grad():
            model_output = model(input_ids=input_ids, attention_mask=attention_mask)
        
        batch_embeddings = mean_pooling(model_output, attention_mask)
        
        # normalize vector (rất quan trọng cho cosine)
        batch_embeddings = torch.nn.functional.normalize(batch_embeddings, p=2, dim=1)
        
        embeddings.append(batch_embeddings.cpu().numpy())
    
    return np.vstack(embeddings)

In [55]:
job_texts = df_clean["job_text_matching"].fillna("").tolist()

job_embeddings = encode_texts(job_texts, batch_size=16, max_length=256)

print("Job embeddings shape:", job_embeddings.shape)

100%|██████████| 21/21 [01:11<00:00,  3.41s/it]

Job embeddings shape: (325, 768)


In [56]:
df_clean["job_embedding"] = list(job_embeddings)

print("[INFO] Saved job embeddings into dataframe")

[INFO] Saved job embeddings into dataframe


In [73]:
cv_text = """
[TITLE] data scientist
[SKILLS] python; sql; power bi; excel, docker
[EXPERIENCE] 3-5 years
[EDUCATION] bachelor
[DESCRIPTION] phân tích dữ liệu, xây dựng dashboard, xử lý dữ liệu lớn
"""

In [74]:
cv_embedding = encode_texts([cv_text], batch_size=1)[0]

print("CV embedding shape:", cv_embedding.shape)

100%|██████████| 1/1 [00:00<00:00,  9.14it/s]

CV embedding shape: (768,)


In [66]:
def cosine_similarity(vec, mat):
    return np.dot(mat, vec)

In [76]:
scores = cosine_similarity(cv_embedding, job_embeddings)

df_clean["similarity_score"] = scores

In [77]:
top_k = 10

top_jobs = df_clean.sort_values("similarity_score", ascending=False).head(top_k)

display(
    top_jobs[
        [
            "job_title_raw",
            "job_title_norm",
            "skills_text",
            "location_norm",
            "similarity_score"
        ]
    ]
)

,job_title_raw,job_title_norm,skills_text,location_norm,similarity_score
146,Data Analyst,data analyst,python; sql; etl; data warehouse; airflow; spa...,hồ chí minh,0.939317
190,Database Engineer,database engineer,python; scala; sql; etl; data lake; airflow; s...,hưng yên,0.933869
283,Senior Business Data Analyst Tại UpBase_Tech-D...,senior business data analyst tại upbase_tech-d...,sql; data warehouse; power bi; tableau; looker...,hà nội,0.931197
166,Data Engineer,data engineer,python; sql; mongodb; airflow; spark; hadoop; ...,hà nội,0.929150
230,Nhân Viên Data Engineer - Đi Làm Ngay,nhân viên data engineer - đi làm ngay,python; sql; etl; data warehouse; airflow; spa...,hà nội,0.928389
159,Data Engineer,data engineer,python; java; scala; etl; data warehouse; spar...,hà nội,0.927421
155,Data Engineer ( Java / Python /Scala/Hadoop/Ka...,data engineer,python; java; scala; hadoop; data lake; spark;...,hà nội,0.926843
168,Data Engineer,data engineer,python; sql; etl; spark; aws; airflow; machine...,hà nội,0.923300
167,Data Engineer,data engineer,python; sql; etl; spark; aws; data quality; azure,hà nội,0.923277
158,Data Engineer Từ 2 Năm Kinh Nghiệm,data engineer từ 2 năm kinh nghiệm,python; r; java; scala; sql; mongodb; etl; air...,hà nội,0.922694


In [78]:
def compute_skill_overlap(cv_skills, job_skills):
    if not cv_skills or not job_skills:
        return 0
    
    cv_set = set(cv_skills)
    job_set = set(job_skills)
    
    return len(cv_set & job_set) / len(job_set)

In [79]:
cv_skills = ["python", "sql", "power bi", "excel"]

df_clean["skill_score"] = df_clean["skills_extracted"].apply(
    lambda x: compute_skill_overlap(cv_skills, x)
)

df_clean["final_score"] = (
    0.7 * df_clean["similarity_score"] +
    0.3 * df_clean["skill_score"]
)

In [81]:
top_jobs = df_clean.sort_values("final_score", ascending=False).head(10)

display(
    top_jobs[
        [
            "job_title_raw",
            "skills_text",
            "location_norm",
            "similarity_score",
            "skill_score",
            "final_score"
        ]
    ]
)

,job_title_raw,skills_text,location_norm,similarity_score,skill_score,final_score
143,Data Analyst,sql,hà nội,0.907745,1.0,0.935422
147,Data Analyst,python,hồ chí minh,0.903797,1.0,0.932658
58,AI Engineer,python; sql,cần thơ,0.903657,1.0,0.932560
310,Thực Tập Sinh Thương Mại,excel,hồ chí minh,0.902292,1.0,0.931604
132,Data Analyst - App Mobile (Junior),sql; power bi; excel,hà nội,0.892026,1.0,0.924418
250,Nhân Viên Nhập Và Xử Lý Dữ Liệu Tiếng Nhật N3 ...,excel,hà nội,0.891161,1.0,0.923813
138,Data Analyst Intern,excel,hồ chí minh,0.890459,1.0,0.923321
96,Chuyên Viên Phân Tích Dữ Liệu (Hàng May Mặc),excel,hưng yên,0.889924,1.0,0.922947
90,Chuyên Viên Dữ Liệu Tài Chính,power bi; excel,hà nội,0.889893,1.0,0.922925
99,Chuyên Viên Phân Tích Dữ Liệu,sql; excel,hà nội,0.877897,1.0,0.914528


# 2. Chuẩn hóa các cột

## - Extract khu vực

In [42]:
# DANH SÁCH THÀNH PHỐ / TỈNH
VIETNAM_CITIES = {
    # TP. Hà Nội
    "hà nội", "ha noi", "tp hà nội", "tp ha noi",
    # TP. Hồ Chí Minh
    "hồ chí minh", "ho chi minh", "tp hồ chí minh", "tp ho chi minh",
    # TP. Hải Phòng
    "hải phòng", "hai phong", "tp hải phòng", "tp hai phong",
    # TP. Huế
    "Huế", "hue", "tp huế", "tp hue",
    # TP. Đà Nẵng
    "đà nẵng", "da nang", "tp đà nẵng", "tp da nang",
    # TP. Cần Thơ
    "cần thơ", "can tho", "tp cần thơ", "tp can tho",
    
    "an giang", "an giang",
    "bắc ninh", "bac ninh",
    "cà mau", "ca mau",
    "cao bằng", "cao bang",
    "đắk lắk", "dak lak",
    "điện biên", "dien bien",
    "đồng nai", "dong nai",
    "đồng tháp", "dong thap",
    "gia lai", "gia lai",
    "hà tĩnh", "ha tinh",
    "hưng yên", "hung yen",
    "khánh hòa", "khanh hoa",
    "lai châu", "lai chau",
    "lâm đồng", "lam dong",
    "lạng sơn", "lang son",
    "lào cai", "lao cai",
    "nghệ an", "nghe an",
    "ninh bình", "ninh binh",
    "phú thọ", "phu tho",
    "quảng ngãi", "quang ngai",
    "quảng ninh", "quang ninh"
    "quảng trị", "quang tri",
    "sơn la", "son la",
    "tây ninh", "tay ninh",
    "thái nguyên", "thai nguyen",
    "thanh hóa", "thanh hoa",
    "tuyên quang", "tuyen quang",
    "vĩnh long", "vinh long",
}

# =========================
# HÀM TẠO clean location
# =========================
def clean_location(location_raw: str, working_addresses_raw: str) -> str:
    if pd.isna(location_raw) or not str(location_raw).strip():
        return ""
    
    loc = str(location_raw).strip()
    
    # Kiểm tra có nhiều địa điểm không
    multi_keywords = {"và", "&", "or", ",", "với", "cùng", "và các"}
    has_multiple = any(kw in loc.lower() for kw in multi_keywords)
    
    # Trường hợp chỉ 1 địa điểm → trả về nguyên bản
    if not has_multiple:
        return loc
    
    # Trường hợp nhiều địa điểm → trích xuất tất cả từ working_addresses_raw
    if pd.isna(working_addresses_raw) or not str(working_addresses_raw).strip():
        return loc  # fallback
    
    addr = str(working_addresses_raw).lower()   # so sánh không phân biệt hoa thường
    
    found_cities = []
    for city in VIETNAM_CITIES:
        if city in addr:
            # Lấy phiên bản có dấu đẹp để hiển thị
            found_cities.append(city)
    
    if found_cities:
        # Loại bỏ trùng lặp và sắp xếp theo thứ tự xuất hiện trong danh sách gốc
        seen = set()
        unique_cities = []
        for city in found_cities:
            if city not in seen:
                seen.add(city)
                unique_cities.append(city)
        
        return ", ".join(unique_cities)
    
    # Nếu không tìm thấy thành phố nào trong danh sách
    return loc

df_clean["location_clean"] = df_clean.apply(
    lambda row: clean_location(
        row.get("location_raw", ""), 
        row.get("working_addresses_raw", "") or row.get("working_addresses_clean", "")
    ), 
    axis=1
)

In [43]:
# Kiểm tra kết quả
print("=== Kết quả location_clean ===")
print(f"Tổng số giá trị duy nhất: {df_clean['location_clean'].nunique()}\n")

display(df_clean[["location_raw", "working_addresses_raw", "location_clean"]].head(15))

# Xem các trường hợp có nhiều địa điểm
multi = df_clean[df_clean["location_raw"].str.contains("và|&|or|,", na=False, case=False)]
print(f"\nSố job có nhiều địa điểm: {len(multi)}")
if len(multi) > 0:
    display(multi[["location_raw", "location_clean"]].head(10))

=== Kết quả location_clean ===
Tổng số giá trị duy nhất: 14



,location_raw,working_addresses_raw,location_clean
0,hồ chí minh,(đã được cập nhật theo Danh mục Hành chính mới...,hồ chí minh
1,hà nội,(đã được cập nhật theo Danh mục Hành chính mới...,hà nội
2,hà nội,(đã được cập nhật theo Danh mục Hành chính mới...,hà nội
3,hà nội,(đã được cập nhật theo Danh mục Hành chính mới...,hà nội
4,hà nội,(đã được cập nhật theo Danh mục Hành chính mới...,hà nội
5,hà nội,(đã được cập nhật theo Danh mục Hành chính mới...,hà nội
6,hà nội,(đã được cập nhật theo Danh mục Hành chính mới...,hà nội
7,hà nội,(đã được cập nhật theo Danh mục Hành chính mới...,hà nội
8,hồ chí minh,(đã được cập nhật theo Danh mục Hành chính mới...,hồ chí minh
9,hồ chí minh,(đã được cập nhật theo Danh mục Hành chính mới...,hồ chí minh



Số job có nhiều địa điểm: 9


,location_raw,location_clean
48,"hà nội , hồ chí minh","hồ chí minh, hà nội"
63,hà nội và 18 nơi khác,"gia lai, cần thơ, hồ chí minh, đồng nai, đồng ..."
64,"hà nội , hồ chí minh","hồ chí minh, hà nội"
115,"hà nội , phú thọ","phú thọ, hà nội"
181,"hà nội , hồ chí minh","hồ chí minh, hà nội"
295,hồ chí minh và 2 nơi khác,"hồ chí minh, đà nẵng, hà nội"
306,hồ chí minh và 3 nơi khác,"hồ chí minh, khánh hòa, đà nẵng, hà nội"
309,"hà nội , hồ chí minh","hồ chí minh, hà nội"
319,"hà nội , phú thọ","phú thọ, hà nội"


In [44]:
# pd.set_option('display.max_columns', None)   # hiển thị tất cả cột

In [45]:
# pd.set_option('display.max_rows', None)      # hiển thị tất cả dòng

In [46]:
# pd.set_option('display.width', None)         # không giới hạn chiều rộng

In [47]:
# pd.set_option('display.max_colwidth', None)  # không cắt nội dung cột

## - Chuẩn hóa địa chỉ làm việc (working_addresses)

In [48]:
def clean_working_addresses_solution_3(raw_text, vietnam_cities):
    import re
    import unicodedata
    import pandas as pd

    DISTRICT_PATTERN = re.compile(
        r'(?i)\b(quận|huyện|thị xã|thành phố)\s+[^\n,;()]+'
    )

    FLOOR_PATTERN = re.compile(
        r'(?i)\b(?:tầng|lầu|floor|fl)\s*[a-z0-9,\-–/ ]+'
    )

    def normalize_empty_value(val):
        if pd.isna(val):
            return None
        val_str = str(val).strip()
        if not val_str or val_str.lower() in {"nan", "none", "null"}:
            return None
        return val_str

    def normalize_unicode(text):
        if text is None:
            return ""
        return unicodedata.normalize("NFC", str(text))

    def clean_address_text(text):
        text = normalize_empty_value(text)
        if text is None:
            return ""
        text = normalize_unicode(text)
        text = text.replace("\\n", "\n")
        text = re.sub(r"[ \t]+", " ", text)
        text = re.sub(r"\s*\n\s*", "\n", text)
        return text.strip()

    def remove_leading_note(text):
        text = clean_address_text(text)
        if not text:
            return ""
        return re.sub(r'^\s*\([^)]*\)\s*', '', text).strip()

    def detect_city_from_text(text):
        text = clean_address_text(text)
        if not text:
            return None
        text_lower = text.lower()
        for city in vietnam_cities:
            city_norm = normalize_empty_value(city)
            if city_norm and city_norm.lower() in text_lower:
                return city_norm
        return None

    def extract_district_note(text):
        text = clean_address_text(text)
        if not text:
            return None
        matches = re.findall(r'\(([^)]*)\)', text)
        matches = [m.strip() for m in matches if normalize_empty_value(m)]
        return "; ".join(matches) if matches else None

    def remove_parentheses_content(text):
        text = clean_address_text(text)
        if not text:
            return ""
        return re.sub(r'\([^)]*\)', '', text).strip()

    def classify_chunk(chunk):
        chunk_lower = chunk.lower().strip()

        if re.search(r'\b(phường|xã|thị trấn)\b', chunk_lower):
            return "ward"

        if re.search(r'\b(tầng|lầu|floor|fl)\b', chunk_lower):
            return "floor_or_building"

        if re.search(r'\b(?:tòa|toà|tower|building|plaza|center|centre|complex|landmark|office)\b', chunk_lower):
            return "building"

        if re.search(r'\b(?:số\s*)?\d+[a-z0-9/\-]*\s+', chunk_lower):
            return "street"

        return "other"

    def parse_floor_and_building_from_chunk(chunk):
        floor_match = FLOOR_PATTERN.search(chunk)
        floor = floor_match.group(0).strip() if floor_match else None

        building = chunk
        if floor:
            building = FLOOR_PATTERN.sub("", building).strip(" ,-")
        if not building:
            building = None

        return floor, building

    def empty_result():
        return {
            "city": None,
            "ward": None,
            "district": None,
            "district_note": None,
            "floor": None,
            "building": None,
            "street_address": None,
            "location_detail": None,
            "full_parsed": None,
            "parse_method": "solution_3"
        }

    raw_text = normalize_empty_value(raw_text)
    if raw_text is None:
        return empty_result()

    text = clean_address_text(raw_text)
    text = remove_leading_note(text)
    text = re.sub(r'^\-\s*', '', text).strip()

    if not text:
        return empty_result()

    city = None
    detail = text

    if ":" in text:
        left, right = text.split(":", 1)
        detail = right.strip()
        city = detect_city_from_text(left.strip())

    if not city:
        city = detect_city_from_text(text)

    district_note = extract_district_note(detail)
    detail_no_note = remove_parentheses_content(detail)

    district_match = DISTRICT_PATTERN.search(detail_no_note)
    district = district_match.group(0).strip() if district_match else None

    chunks = [c.strip() for c in detail_no_note.split(",") if c.strip()]

    ward = None
    floor = None
    building = None
    street_address = None
    other_chunks = []

    for chunk in chunks:
        label = classify_chunk(chunk)

        if label == "ward" and ward is None:
            ward = chunk

        elif label == "floor_or_building":
            chunk_floor, chunk_building = parse_floor_and_building_from_chunk(chunk)
            if floor is None and chunk_floor:
                floor = chunk_floor
            if chunk_building:
                building = f"{building}, {chunk_building}" if building else chunk_building

        elif label == "building":
            building = f"{building}, {chunk}" if building else chunk

        elif label == "street" and street_address is None:
            street_address = chunk

        else:
            other_chunks.append(chunk)

    location_detail_parts = []
    if floor:
        location_detail_parts.append(floor)
    if building:
        location_detail_parts.append(building)
    if street_address:
        location_detail_parts.append(street_address)
    if other_chunks:
        location_detail_parts.extend(other_chunks)

    location_detail = ", ".join(location_detail_parts) if location_detail_parts else None

    full_parts = [street_address or building or location_detail, ward, city]
    full_parsed = ", ".join([p for p in full_parts if p]) if any(full_parts) else None

    return {
        "city": city,
        "ward": ward,
        "district": district,
        "district_note": district_note,
        "floor": floor,
        "building": building,
        "street_address": street_address,
        "location_detail": location_detail,
        "full_parsed": full_parsed,
        "parse_method": "solution_3"
    }

In [49]:
parsed_s3 = df["working_addresses_raw"].apply(
    lambda x: clean_working_addresses_solution_3(x, VIETNAM_CITIES)
)
df_s3 = pd.concat([df, pd.json_normalize(parsed_s3).add_prefix("s3_")], axis=1)

In [50]:
display(df_s3[["working_addresses_raw", "s3_full_parsed"]].head(10))

,working_addresses_raw,s3_full_parsed
0,(đã được cập nhật theo Danh mục Hành chính mới...,"TTC 253 Hoàng Văn Thụ, Phường Tân Sơn Hòa, hồ ..."
1,(đã được cập nhật theo Danh mục Hành chính mới...,"61 Ngụy Như Kon Tum, Phường Thanh Xuân \nHà Nộ..."
2,(đã được cập nhật theo Danh mục Hành chính mới...,"6 Tòa Comatce Tower - 61 Ngụy Như Kon Tum, Phư..."
3,(đã được cập nhật theo Danh mục Hành chính mới...,"số 10 Phạm Văn Bạch, Phường Cầu Giấy, hà nội"
4,(đã được cập nhật theo Danh mục Hành chính mới...,"Landmark 72, Phường Yên Hòa, hà nội"
5,(đã được cập nhật theo Danh mục Hành chính mới...,"Keangnam Landmark 72, Phường Yên Hòa, hà nội"
6,(đã được cập nhật theo Danh mục Hành chính mới...,"Keangnam Landmark 72, Phường Yên Hòa, hà nội"
7,(đã được cập nhật theo Danh mục Hành chính mới...,"số 10 Phạm Văn Bạch, Xã Hòa Lạc - Hà Nội: FPT..."
8,(đã được cập nhật theo Danh mục Hành chính mới...,"20 Cộng Hòa, Phường Bảy Hiền, hồ chí minh"
9,(đã được cập nhật theo Danh mục Hành chính mới...,"20 Cộng Hòa, Phường Bảy Hiền, hồ chí minh"


In [51]:
def clean_working_addresses(raw_text):
    if pd.isna(raw_text) or not str(raw_text).strip():
        return []

    text = str(raw_text).strip()

    # 1) bỏ note đầu chuỗi trong ngoặc nếu có
    text = re.sub(r'^\s*\([^)]*\)\s*', '', text).strip()

    # 2) chuẩn hóa xuống dòng
    text = text.replace('\\n', '\n')

    # 3) tách từng dòng có địa chỉ
    lines = [line.strip() for line in text.split('\n') if line.strip()]

    results = []

    for line in lines:
        # bỏ dấu đầu dòng "-"
        line = re.sub(r'^\-\s*', '', line).strip()

        # 4) tách city theo dấu :
        city = None
        detail = line

        if ':' in line:
            left, right = line.split(':', 1)
            city = left.strip()
            detail = right.strip()

        # 5) lấy note trong ngoặc ở cuối
        note_matches = re.findall(r'\(([^)]*)\)', detail)
        district_note = '; '.join(note_matches) if note_matches else None

        # bỏ phần ngoặc ra khỏi detail
        detail_clean = re.sub(r'\([^)]*\)', '', detail).strip()

        # 6) split theo dấu phẩy
        parts = [p.strip() for p in detail_clean.split(',') if p.strip()]

        street = None
        ward = None
        district = None

        if len(parts) >= 1:
            street = parts[0]

        for p in parts[1:]:
            p_lower = p.lower()
            if p_lower.startswith(('tầng', 'floor', 'tòa')):
                location_detail = p
            elif p_lower.startswith(('phường', 'xã', 'thị trấn')):
                ward = p
            elif p_lower.startswith(('quận', 'huyện', 'thị xã', 'thành phố')):
                district = p
            elif street is None:
                street = p

        results.append({
            'city': city,
            'street': street,
            'ward': ward,
            'location_detail': location_detail if 'location_detail' in locals() else None,
            'district': district,
            'district_note': district_note,
            'full_parsed': f"{street}, {ward}, {city}" if street and ward and city else None
        })

    return results

In [52]:
df_clean["working_addresses_clean"] = df_clean["working_addresses_raw"].apply(clean_working_addresses)

In [53]:
def clean_working_addresses(text: str) -> str:
    """
    Clean working_addresses_raw thành địa chỉ hoàn chỉnh, dễ đọc.
    """
    if pd.isna(text) or not str(text).strip():
        return ""
    
    text = str(text).strip()
    
    # Bước 1: Loại bỏ phần mở đầu dài dòng (phần trong ngoặc đầu tiên)
    text = re.sub(r'^\s*\(đã được cập nhật theo danh mục hành chính mới.*?\)\s*[-–—]\s*', '', text, flags=re.IGNORECASE)
    
    # Bước 2: Loại bỏ các cụm ghi chú thừa còn lại trong ngoặc
    text = re.sub(r'\s*\([^)]*quận/huyện cũ tương ứng.*?\)', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\s*\([^)]*dễ dàng tra cứu.*?\)', '', text, flags=re.IGNORECASE)
    
    # Bước 3: Loại bỏ tất cả nội dung trong ngoặc đơn (an toàn)
    text = re.sub(r'\([^)]*\)', '', text)
    
    # Bước 4: Chuẩn hóa dấu phân cách (thay - , ; bằng dấu phẩy)
    text = re.sub(r'\s*[-–—;,]\s*', ', ', text)
    
    # Bước 5: Sử dụng clean_text để lowercase + chuẩn hóa khoảng trắng
    text = clean_text(text)
    
    # Bước 6: Viết hoa chữ cái đầu mỗi thành phần (tùy chọn nhưng nên làm cho đẹp)
    def capitalize_parts(s):
        parts = [p.strip() for p in s.split(',') if p.strip()]
        capitalized = [p.capitalize() if not p[0].isdigit() else p for p in parts]
        return ', '.join(capitalized)
    
    text = capitalize_parts(text)
    
    return text


# ==================== ÁP DỤNG ====================
df_clean["working_addresses_clean"] = df_clean["working_addresses_raw"].apply(clean_working_addresses)

# Kiểm tra kết quả
print("Kết quả sau khi clean working_addresses:")
display(df_clean[["working_addresses_raw", "working_addresses_clean"]].head(1))

Kết quả sau khi clean working_addresses:


,working_addresses_raw,working_addresses_clean
0,(đã được cập nhật theo Danh mục Hành chính mới...,"Hồ chí minh ttc 253 hoàng văn thụ, Phường tân ..."


In [54]:
df_clean[["job_title_raw", "working_addresses_raw", "working_addresses_clean"]].head(10)

,job_title_raw,working_addresses_raw,working_addresses_clean
0,junior fp a analyst hồ chí minh,(đã được cập nhật theo Danh mục Hành chính mới...,"Hồ chí minh ttc 253 hoàng văn thụ, Phường tân ..."
1,data governance specialist,(đã được cập nhật theo Danh mục Hành chính mới...,"Hà nội tầng 2, 3, 5, 6 tòa comatce, 61 ngụy nh..."
2,project manager dự án ai hub,(đã được cập nhật theo Danh mục Hành chính mới...,"Hà nội tầng 2, 3, 5, 6 tòa comatce tower, 61 n..."
3,ai engineer,(đã được cập nhật theo Danh mục Hành chính mới...,"Hà nội fpt tower, Số 10 phạm văn bạch, Phường ..."
4,ai engineer,(đã được cập nhật theo Danh mục Hành chính mới...,"Hà nội landmark 72, Keangnam phạm hùng, Phường..."
5,data analyst,(đã được cập nhật theo Danh mục Hành chính mới...,"Hà nội keangnam landmark 72, E6 phạm hùng, Phư..."
6,data engineer,(đã được cập nhật theo Danh mục Hành chính mới...,"Hà nội keangnam landmark 72, Phường yên hòa"
7,fresher data engineer,(đã được cập nhật theo Danh mục Hành chính mới...,"Hà nội khu công nghệ cao hòa lạc, Xã hòa lạc, ..."
8,data analyst teamleader collection analytics,(đã được cập nhật theo Danh mục Hành chính mới...,"Hồ chí minh 20 cộng hòa, Phường bảy hiền"
9,data analyst teamleader collection analytics t...,(đã được cập nhật theo Danh mục Hành chính mới...,"Hồ chí minh 20 cộng hòa, Phường bảy hiền"


## - Chuẩn hóa lương (salary)

In [55]:
def clean_salary(raw: str) -> str:
    text = normalize_empty_value(raw)
    if text is None:
        return ""
    
    text = normalize_unicode(text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    
    # Chuẩn hóa khoảng trắng quanh dấu gạch nối
    text = re.sub(r"\s*-\s*", " - ", text)
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [56]:
df_clean["salary_clean"] = df_clean["salary_raw"].apply(clean_salary)

## - Chuẩn hóa kinh nghiệm

In [57]:
def clean_experience(text: str):
    text = normalize_empty_value(text)
    if text is None:
        return None
    
    text = normalize_unicode(text).lower().strip()

    # Không yêu cầu
    if "không yêu cầu" in text:
        return 0

    # Dưới 1 năm
    if "dưới 1 năm" in text:
        return 0

    # Lấy tất cả số
    nums = [int(x) for x in re.findall(r"\d+", text)]
    if not nums:
        return None

    # Nếu là khoảng thì lấy min
    return nums[0]

In [58]:
df_clean["experience_clean"] = df_clean["experience_raw"].apply(clean_experience)

## - Extract tag/skill

In [59]:
SYNONYM_MAP = {
    # role
    "business analyst (phân tích nghiệp vụ)": "business analyst",

    # domain
    "giáo dục / đào tạo": "giáo dục",
    "y tế / dược phẩm": "y tế",
    "logistic / vận tải": "logistics",
    "bán lẻ - hàng tiêu dùng - fmcg": "fmcg",
    "marketing / quảng cáo": "marketing",

    # tech
    "data labeling (gán nhãn dữ liệu)": "data labeling",
    "điện toán đám mây (cloud)": "cloud",

    # giữ nguyên canonical nếu đã chuẩn
    "data analyst": "data analyst",
    "data engineer": "data engineer",
    "data scientist": "data scientist",
    "ai engineer": "ai engineer",
    "ai researcher": "ai researcher",
    "phân tích dữ liệu": "phân tích dữ liệu",
    "an ninh mạng": "an ninh mạng",
    "it - phần mềm": "it - phần mềm",
    "it - phần cứng và máy tính": "it - phần cứng và máy tính",
    "ngân hàng": "ngân hàng",
    "tài chính": "tài chính",
    "kế toán": "kế toán",
    "thương mại điện tử": "thương mại điện tử",
    "viễn thông": "viễn thông",
    "game": "game",
    "chứng khoán": "chứng khoán",
    "công nghệ kỹ thuật": "công nghệ kỹ thuật",
    "công nghệ thông tin khác": "công nghệ thông tin khác",
}

In [60]:
ROLE_KEYWORDS = {
    "data analyst",
    "data engineer",
    "data scientist",
    "ai engineer",
    "ai researcher",
    "business analyst",
}

DOMAIN_KEYWORDS = {
    "ngân hàng",
    "tài chính",
    "kế toán",
    "thương mại điện tử",
    "giáo dục",
    "fmcg",
    "marketing",
    "y tế",
    "logistics",
    "viễn thông",
    "chứng khoán",
    "game",
    "may mặc / thời trang",
    "luật / pháp chế",
}

TECH_KEYWORDS = {
    "it - phần mềm",
    "it - phần cứng và máy tính",
    "phân tích dữ liệu",
    "data labeling",
    "cloud",
    "an ninh mạng",
    "công nghệ kỹ thuật",
    "công nghệ thông tin khác",
}

In [61]:
def normalize_tag(tag: str) -> str:
    tag = normalize_empty_value(tag)
    if tag is None:
        return ""

    tag = normalize_unicode(tag).lower()
    tag = re.sub(r"<[^>]+>", " ", tag)
    tag = re.sub(r"\s+", " ", tag).strip()

    # bỏ prefix
    tag = re.sub(r"^chuyên môn\s+", "", tag).strip()

    # map về canonical name
    tag = SYNONYM_MAP.get(tag, tag)

    return tag

In [62]:
IGNORE_TAGS = {"khác"}

def extract_tags_info(raw: str):
    text = normalize_empty_value(raw)
    if text is None:
        return {
            "tags_clean": [],
            "tags_norm": [],
            "primary_role": None,
            "domain_tags": [],
            "tech_tags": [],
        }

    text = normalize_unicode(text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    parts = re.split(r"[;|,]+", text)

    tags_clean = []
    tags_norm = []
    seen_clean = set()
    seen_norm = set()

    for part in parts:
        clean = part.strip()
        if not clean:
            continue

        norm = normalize_tag(clean)
        if not norm or norm in IGNORE_TAGS:
            continue

        key_clean = clean.lower()
        if key_clean not in seen_clean:
            seen_clean.add(key_clean)
            tags_clean.append(clean)

        if norm not in seen_norm:
            seen_norm.add(norm)
            tags_norm.append(norm)

    primary_role = None
    domain_tags = []
    tech_tags = []

    for tag in tags_norm:
        if primary_role is None and tag in ROLE_KEYWORDS:
            primary_role = tag
            continue

        if tag in DOMAIN_KEYWORDS:
            domain_tags.append(tag)
            continue

        if tag in TECH_KEYWORDS:
            tech_tags.append(tag)
            continue

    return {
        "tags_clean": tags_clean,
        "tags_norm": tags_norm,
        "primary_role": primary_role,
        "domain_tags": domain_tags,
        "tech_tags": tech_tags,
    }

In [63]:
df_clean["primary_role"] = df["tags_raw"].apply(lambda x: extract_tags_info(x)["primary_role"])
df_clean["domain_tags"] = df["tags_raw"].apply(lambda x: extract_tags_info(x)["domain_tags"])
df_clean["tech_tags"] = df["tags_raw"].apply(lambda x: extract_tags_info(x)["tech_tags"])
df_clean["tags_norm"] = df["tags_raw"].apply(lambda x: extract_tags_info(x)["tags_norm"])

In [64]:
df_clean[["job_title_raw","tags_raw", "tags_norm", "primary_role", "domain_tags", "tech_tags"]].head()

,job_title_raw,tags_raw,tags_norm,primary_role,domain_tags,tech_tags
0,junior fp a analyst hồ chí minh,Chuyên môn Data Analyst; Tài chính; Kế toán,"[data analyst, tài chính, kế toán]",data analyst,"[tài chính, kế toán]",[]
1,data governance specialist,Chuyên môn Data Analyst,[data analyst],data analyst,[],[]
2,project manager dự án ai hub,Chuyên môn IT Project Manager,[it project manager],NaN,[],[]
3,ai engineer,Chuyên môn AI Engineer; IT - Phần mềm,"[ai engineer, it - phần mềm]",ai engineer,[],[it - phần mềm]
4,ai engineer,Chuyên môn AI Engineer,[ai engineer],ai engineer,[],[]


## - Extract skill từ text

In [ ]:
# Dictionary mapping skill
SKILL_DICT = {
    "python": ["python"],
    "sql": ["sql", "mysql", "postgresql", "sql server", "mssql"],
    "excel": ["excel", "microsoft excel"],
    "power_bi": ["power bi", "powerbi"],
    "tableau": ["tableau"],
    "pandas": ["pandas"],
    "numpy": ["numpy"],
    "scikit_learn": ["scikit-learn", "sklearn"],
    "machine_learning": ["machine learning", "ml"],
    "deep_learning": ["deep learning", "dl"],
    "spark": ["spark", "apache spark"],
    "hadoop": ["hadoop"],
    "airflow": ["airflow"],
    "docker": ["docker"],
    "git": ["git", "github", "gitlab"],
    "statistics": ["thống kê", "statistics", "statistical"],
    "data_visualization": ["trực quan hóa dữ liệu", "data visualization", "visualization"],
}

print("Số skill trong dict:", len(SKILL_DICT))
print("Các skill:", list(SKILL_DICT.keys()))

In [71]:
def extract_skills(text: str, skill_dict: dict) -> list:
    text = clean_text(text)
    found = set()
    for canonical, aliases in skill_dict.items():
        for alias in aliases:
            alias_clean = clean_text(alias)
            pattern = rf"(?<!\w){re.escape(alias_clean)}(?!\w)"
            if re.search(pattern, text, re.IGNORECASE):
                found.add(canonical)
                break
    return sorted(found)


# Nguồn text để extract skill
skill_source = (
    df["job_title_raw"].fillna("") + " " +
    df["requirements_raw"].fillna("") + " " +
    df["description_raw"].fillna("")
)

df["skills_normalized"] = skill_source.apply(lambda x: extract_skills(x, SKILL_DICT))
df["skills_normalized_str"] = df["skills_normalized"].apply(lambda x: " | ".join(x) if x else "")

# Kiểm tra
print("Top 15 skill phổ biến:")
skill_flat = [s for sublist in df["skills_normalized"] for s in sublist]
pd.Series(skill_flat).value_counts().head(15)

Top 15 skill phổ biến:


python                148
sql                   139
machine_learning       99
power_bi               61
statistics             54
excel                  48
tableau                47
spark                  45
docker                 45
data_visualization     40
deep_learning          40
git                    34
airflow                30
scikit_learn           25
hadoop                 23
Name: count, dtype: int64

In [72]:
def build_job_text_for_phobert(row):
    parts = []

    # Trong pipeline hiện có các cột đã được clean như:
    #   - job_title_raw (đã clean)  
    #   - description_clean  
    #   - requirements_clean  
    #   - experience_clean  
    # Vì vậy chúng ta dùng những cột này để tránh cột bị thiếu.

    if row.get("job_title_raw"):
        parts.append(f"[JOB_TITLE] {row['job_title_raw']}")

    if row.get("tags_norm"):
        tags = ", ".join(row["tags_norm"]) if isinstance(row["tags_norm"], list) else str(row["tags_norm"])
        parts.append(f"[TAGS] {tags}")

    if row.get("experience_clean") is not None:
        parts.append(f"[EXPERIENCE] {row['experience_clean']} năm")

    if row.get("location_clean"):
        parts.append(f"[LOCATION] {row['location_clean']}")

    if row.get("description_clean"):
        parts.append(f"[DESCRIPTION] {row['description_clean']}")

    if row.get("requirements_clean"):
        parts.append(f"[REQUIREMENTS] {row['requirements_clean']}")

    return "\n".join(parts)

In [73]:
# Sử dụng df_clean vì các cột đã được clean (location_clean, description_clean, requirements_clean)
df_clean["jd_full_text"] = df_clean.apply(build_job_text_for_phobert, axis=1)

In [75]:
pd.set_option('display.width', None)         # không giới hạn chiều rộng
pd.set_option('display.max_colwidth', None)  # không cắt nội dung cột

In [76]:
df_clean[["job_title_raw", "jd_full_text"]].head(3)

,job_title_raw,jd_full_text
0,junior fp a analyst hồ chí minh,"[JOB_TITLE] junior fp a analyst hồ chí minh\n[TAGS] data analyst, tài chính, kế toán\n[EXPERIENCE] 2 năm\n[LOCATION] hồ chí minh"
1,data governance specialist,[JOB_TITLE] data governance specialist\n[TAGS] data analyst\n[EXPERIENCE] 2 năm\n[LOCATION] hà nội
2,project manager dự án ai hub,[JOB_TITLE] project manager dự án ai hub\n[TAGS] it project manager\n[EXPERIENCE] 2 năm\n[LOCATION] hà nội


In [ ]:
def select_final_columns(df):
    final_cols = [
        "job_url", "source_field_name", "field_count",
        "job_title_raw", "job_title_clean",
        "company_name_raw", "company_url",
        "salary_raw", "salary_min", "salary_max", "salary_type",
        "location_raw", "location_normalized",
        "experience_raw", "experience_min_years", "experience_max_years", "experience_type",
        "job_level_raw", "education_level_raw", "employment_type_raw", "deadline_raw",
        "company_scale_raw", "company_address_raw", "company_description_raw",
        "description_raw", "description_clean",
        "requirements_raw", "requirements_clean",
        "benefits_raw", "benefits_clean",
        "skills_normalized", "skills_normalized_str",
        "job_text"
    ]
    existing = [c for c in final_cols if c in df.columns]
    return df[existing].copy()


final_df = select_final_columns(df)
print("Final shape:", final_df.shape)
display(final_df.head(3))

In [ ]:
def save_processed_data(df, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    ts = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    
    csv_path = os.path.join(output_dir, f"jobs_nlp_ready_{ts}.csv")
    xlsx_path = os.path.join(output_dir, f"jobs_nlp_ready_{ts}.xlsx")
    
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"Saved CSV → {csv_path}")
    
    try:
        df.to_excel(xlsx_path, index=False)
        print(f"Saved XLSX → {xlsx_path}")
    except Exception as e:
        print(f"Không lưu được Excel: {e}")


# Lưu
save_processed_data(final_df, OUTPUT_DIR)

In [ ]:
# Export nhanh để test (không timestamp)
quick_path = os.path.join(OUTPUT_DIR, "jobs_nlp_ready_latest_debug.csv")
final_df.to_csv(quick_path, index=False, encoding="utf-8-sig")
print("Quick debug file:", quick_path)

In [ ]:
df_clean.info()

## - Tạo df_final

In [66]:
df_final = pd.DataFrame()
df_final["job_url"] = df_clean["job_url"]
df_final["job_title"] = df_clean["job_title_raw"]

# df_final["tags"] = df_clean["tags_raw"]
df_final["primary_role"] = df_clean["primary_role"]
df_final["domain_tags"] = df_clean["domain_tags"]
df_final["tech_tags"] = df_clean["tech_tags"]
df_final["tags_norm"] = df_clean["tags_norm"]

df_final["salary"] = df_clean["salary_clean"]
df_final["location"] = df_clean["location_clean"]

df_final["working_addresses"] = df_clean["working_addresses_clean"]
# df_final["working_times"] = df_clean["working_times_clean"]
df_final["experience"] = df_clean["experience_clean"]

# df_final["job_level"] = df_clean["job_level_clean"]
# df_final["education_level"] = df_clean["education_level_clean"]
# df_final["employment_type"] = df_clean["employment_type_clean"]
# df_final["job_quantity"] = df_clean["job_quantity_clean"]

# df_final["deadline"] = df_clean["deadline_clean"]

# df_final["description"] = df_clean["description_clean"]
# df_final["requirements"] = df_clean["requirements_clean"]
# df_final["benefits"] = df_clean["benefits_clean"]

# df_final["company_name"] = df_clean["company_name_clean"]
# df_final["company_url"] = df_clean["company_url_clean"]
# df_final["company_website"] = df_clean["company_website_clean"]
# df_final["company_scale"] = df_clean["company_scale_clean"]
# df_final["company_field"] = df_clean["company_field_clean"]
# df_final["company_address"] = df_clean["company_address_clean"]
# df_final["company_description"] = df_clean["company_description_clean"]


In [67]:
df_final.head()

,job_url,job_title,primary_role,domain_tags,tech_tags,tags_norm,salary,location,working_addresses,experience
0,https://www.topcv.vn/brand/beeogisticscorporat...,junior fp a analyst hồ chí minh,data analyst,"[tài chính, kế toán]",[],"[data analyst, tài chính, kế toán]",12 - 20 triệu,hồ chí minh,"Hồ chí minh ttc 253 hoàng văn thụ, Phường tân ...",2
1,https://www.topcv.vn/brand/educa/tuyen-dung/da...,data governance specialist,data analyst,[],[],[data analyst],20 - 30 triệu,hà nội,"Hà nội tầng 2, 3, 5, 6 tòa comatce, 61 ngụy nh...",2
2,https://www.topcv.vn/brand/educa/tuyen-dung/pr...,project manager dự án ai hub,NaN,[],[],[it project manager],30 - 35 triệu,hà nội,"Hà nội tầng 2, 3, 5, 6 tòa comatce tower, 61 n...",2
3,https://www.topcv.vn/brand/fptis/tuyen-dung/ai...,ai engineer,ai engineer,[],[it - phần mềm],"[ai engineer, it - phần mềm]",Thoả thuận,hà nội,"Hà nội fpt tower, Số 10 phạm văn bạch, Phường ...",3
4,https://www.topcv.vn/brand/fptis/tuyen-dung/ai...,ai engineer,ai engineer,[],[],[ai engineer],Thoả thuận,hà nội,"Hà nội landmark 72, Keangnam phạm hùng, Phường...",2
